# GNN Powerflow V2.7 — Contingency Analysis

N-1 line-outage **violation assessment**: GNN predictions vs. PyPSA Newton-Raphson truth vs. DC power flow on every contingency case; reference solvers (PyPower AC, PyPSA single/batch, FDLF, pandapower, DC) timed on a subset.

**Input**: `n1_*.json` dataset index + `<tag>_meta.json` sidecar from `GNN_Powerflow_V2.6_DataGen.ipynb`; sweep/run pickles from `GNN_Powerflow_V2.7_Training.ipynb`

**Output**: printed tables; optional CSVs written to a NEW folder (existing datasets are read-only)

**Copy convention**: every function/class in the *Copied library* section is a verbatim copy marked `# [COPY of <notebook>::<function>]`; the origin notebook is the ground truth and `test_contingency_copy_parity.py` reports drift. Normal precedence is DataGen > Training > Analysis by domain. **D13 exception**: the dataset/evaluator pair (`PowerFlowDataset`, `collate_with_ptdf`, evaluator and helpers) is copied from Analysis (user decision D13, 2026-10-14); each by-rule Training copy carries a `PRECEDENCE EXCEPTION` line.

**D3 caveat**: post-outage dispatch is single slack with no Q limits, so PV/slack bus voltages equal their setpoints (voltage violations can only appear at PQ buses).

In [ ]:
# %%
import copy
import logging
import os
import pickle
import random
import time
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict

# Fix for numba >= 0.60: version_version attribute was removed; inject it before pandapower imports it
import sys
print("pandapower already imported:", "pandapower" in sys.modules)
try:
    import numba._version as _nv
    if not hasattr(_nv, 'version_version'):
        import numba as _n; _nv.version_version = _n.__version__
except ImportError:
    pass

%matplotlib inline
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pypsa
import pypowsybl as pp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch, Data, Dataset
# [STSI 141026]:global_mean_pool/global_max_pool added (as in Training cell 1) for the PowerFlowGNN copy
from torch_geometric.nn import GATv2Conv, GCNConv, GraphConv, TransformerConv, global_mean_pool, global_max_pool  # [STSI120526] Added TransformerConv import
from tqdm import tqdm

import urllib

import statsmodels.api as sm
from statsmodels.formula.api import ols


import numba, pandapower as pdp
from pandapower.converter import from_ppc
import pandapower.networks as pn

# Suppress verbose output from PyPSA and dependencies during data generation
for _log in ("pypsa", "pypsa.pf", "pypsa.components", "numexpr", "linopy", "fontTools", "fontTools.subset"):
    logging.getLogger(_log).setLevel(logging.WARNING)
logging.getLogger("numexpr").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import re
from scipy import ndimage

# [STSI 141026]:metrics imports for contingency violation assessment (tasks 089/090)
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, average_precision_score
from scipy.stats import spearmanr
# [STSI 141026]:module-level import from Analysis cell 9, needed by the precompute_bfs_order copy
from collections import deque

In [ ]:
# ── Configurable paths ───────────────────────────────────────────────────────
# Set DATA_ROOT to wherever your data folders live.
# Use "." to keep everything relative to the repo (default).
# Example: DATA_ROOT = r"C:\Users\you\OneDrive\Data_PF_GNN"
import os as _os
DATA_ROOT             = r"C:\Users\STSI\OneDrive - USN\Data_PF_GNN"  
#DATA_ROOT=              r"C:\Users\ssimo\OneDrive - USN\Data_PF_GNN"                                  
GRID_MODEL_FILES      = _os.path.join(DATA_ROOT, "grid_model_files")    # static grid CSV/RAW files
TRAINING_NETWORKS_DIR = _os.path.join(DATA_ROOT, "training_networks_saved")  # dataset JSONs
TRAINING_RESULTS_DIR  = _os.path.join(DATA_ROOT, "training_results_saved")   # sweep result JSONs/CSVs
# [STSI 141026]:verbatim copy of DataGen cell 2 (configurable paths)

In [ ]:
# [STSI 141026]:contingency notebook execution toggles (task 088)
run_contingency_eval = False  # [STSI 191026]:task 095 default off (F11); run the N-1 violation assessment (GNN vs PyPSA NR truth vs DC) on the selected runs
run_ref_timing = False  # [STSI 191026]:task 095 default off (F11); run reference-solver timing (PyPower/FDLF/pandapower/PyPSA/DC) on a subset of cases

## Copied library (verbatim — do not edit; originals are ground truth)

In [ ]:
# [COPY of GNN_Powerflow_V2.6_DataGen::load_dataset_list]
def load_dataset_list(index_path: str) -> list:
    """
    Reload a list of PyPSA networks saved by save_dataset_list.
    """
    import json

    with open(index_path, "r") as f:
        meta = json.load(f)

    save_dir = os.path.dirname(index_path) or "."
    networks: list[pypsa.Network] = []
    for nc_name in meta["files"]:
        nc_path = os.path.join(save_dir, nc_name)
        networks.append(pypsa.Network(nc_path))

    print(f"Loaded {len(networks)} networks from {index_path}")
    return networks


# [COPY of GNN_Powerflow_V2.6_DataGen::load_dataset_subset]
def load_dataset_subset(index_path, positions):
    """Load only `positions` of a load_dataset_list-format index; same path logic as load_dataset_list."""
    with open(index_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    base_dir = os.path.dirname(index_path)
    return [(int(p), meta["files"][p], pypsa.Network(os.path.join(base_dir, meta["files"][p]))) for p in positions]


# [COPY of GNN_Powerflow_V2.6_DataGen::read_n1_sidecar]
def read_n1_sidecar(meta_path):
    """Returns (header_dict_without_rows, pd.DataFrame(rows))."""
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    rows = meta.pop("rows")
    return meta, pd.DataFrame(rows)


# [COPY of GNN_Powerflow_V2.6_DataGen::attach_line_limits]
def attach_line_limits(net, system, real_ratings, synthetic_limits):
    """Write lines.s_limit_pu / lines.limit_source in place (pass a copy). Never touches s_nom/v_nom."""
    s_lim, src = [], []
    for name in net.lines.index:
        if name in real_ratings:
            v, s = real_ratings[name]
        else:
            v, s = synthetic_limits.get((system, name), float("nan")), "synthetic"
        s_lim.append(float(v)); src.append(s)
    net.lines["s_limit_pu"] = s_lim
    net.lines["limit_source"] = src
    return net


# [STSI 141026]:verbatim copies extracted programmatically (task 088); do not edit, originals are ground truth

In [ ]:
# [COPY of GNN_Powerflow_V2.7_Training::PowerFlowGNN]
class PowerFlowGNN(nn.Module):
    """
    Graph Attention Network (GATv2) for power flow prediction.
    Predicts per-bus: [vmag, vang, P, Q]
    Predicts per-edge bilinear PTDF: hedges @ W @ hnodes.T
    """

    def __init__(
        self,
        node_features: int = 7,
        edge_features: int = 4,
        hidden_dim: int = 64, # Hiddem dimensions means the size of the feature vectors that are passed between layers in the GNN. A larger hidden_dim allows the model to capture more complex relationships, but also increases computational cost and risk of overfitting. 64 is a common choice for a balance between expressiveness and efficiency.
        num_layers: int = 3,# Number of GNN layers (graph convolutional layers). More layers allow the model to capture more complex interactions between nodes, but can also lead to over-smoothing where node features become too similar. 3 layers is a common choice for many graph tasks.
        heads: int = 4,# Number of attention heads in GAT layers. Multiple heads allow the model to attend to different aspects of the graph structure simultaneously. 4 heads is a common choice that provides a good balance between expressiveness and computational cost.
        dropout: float = 0.0,
        conv_type: str = "gatv2", # Placeholder for potential future extension to other convolution types
        head_mode:str = "standard", # or with new encoder
        # ── Conv block flags (independent toggles, replace conv_mode) ── [STSI 120526]:refactored conv_mode→independent flags
        use_residual: bool = False,
        norm_type: str = None,         # None | "layer" | "graph"
        activation: str = "leaky_relu",  # "leaky_relu" | "gelu" | "relu" | "elu" | "silu"
        use_dropout: bool = False,
        drop_rate: float = 0.1,
        # ── Deprecated shorthand ──
        conv_mode: str = None,         # if set, overrides individual flags
        # ── Edge Δθ prediction ── [STSI 250526]: added angle_mode to control whether we predict node angles (standard) or edge angle differences (edge_delta)
        angle_mode: str = "node",      # "node" (default), "edge_delta", or "both"  # [STSI 020626]:added "both" for dual-head
        # ── V_mag residual ── [STSI 280526]: predict ΔV from flat baseline instead of absolute V_mag
        vmag_mode: str = "absolute",   # "absolute" (default) or "residual"
        # ── Global graph context ── [STSI 290526]: whole-graph awareness via mean+max pooling
        use_global_pool: bool = False,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.conv_type = conv_type
        self.head_mode = head_mode
        self.angle_mode = angle_mode  # [STSI 250526]:new angle_mode option
        self.vmag_mode = vmag_mode    # [STSI 280526]:residual V_mag prediction
        self.use_global_pool = use_global_pool  # [STSI 290526]
        # ── Deprecated conv_mode shorthand ── [STSI 120526]:backward compat mapping
        if conv_mode is not None:
            _map = {
                "old":            (False, None,    "leaky_relu", False),
                "residual":       (True,  None,    "gelu",       False),
                "res_norm":       (True,  "layer", "gelu",       False),
                "res_norm_drop":  (True,  "layer", "gelu",       True),
            }
            use_residual, norm_type, activation, use_dropout = _map[conv_mode]

        self.use_residual = use_residual
        self.norm_type = norm_type
        self.use_dropout = use_dropout
        self.drop_rate = drop_rate
        self.activation_name = activation
        # 1. Node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # 2. GAT convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(self._make_conv(hidden_dim, hidden_dim, edge_features))

        # 2b. Optional normalization + dropout for conv block [STSI 120526]:independent flags
        if self.norm_type is not None:
            if self.norm_type == "layer":
                self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
            elif self.norm_type == "graph":
                from torch_geometric.nn.norm import GraphNorm
                self.norms = nn.ModuleList([GraphNorm(hidden_dim) for _ in range(num_layers)])
            else:
                raise ValueError(f"Unknown norm_type: {self.norm_type!r}")
        if self.use_dropout:
            self.drop = nn.Dropout(self.drop_rate)


        # 2c. Optional global pooling projection [STSI 290526]: graph-level context → per-node
        if self.use_global_pool:
            self.global_proj = nn.Linear(2 * hidden_dim, hidden_dim)

        # Activation function lookup [STSI 120526]:configurable activation
        _act_map = {
            "leaky_relu": F.leaky_relu,
            "gelu": F.gelu,
            "relu": F.relu,
            "elu": F.elu,
            "silu": F.silu,
        }
        if activation not in _act_map:
            raise ValueError(f"Unknown activation: {activation!r}")
        self.activation_fn = _act_map[activation]

        # 3. Output layers for different predictions
        if head_mode=="standard":
            if angle_mode == "edge_delta":  # [STSI 250526]:no angle head for node prediction
                self.vmag_pred = nn.Linear(hidden_dim, 1)
                self.p_pred = nn.Linear(hidden_dim, 1)
                self.q_pred = nn.Linear(hidden_dim, 1)
            else:
                # one prediction head for each of the properties we want to predict
                self.vmag_pred = nn.Linear(hidden_dim, 1)   # Voltage magnitude
                self.vang_pred = nn.Linear(hidden_dim, 1)   # Voltage angle
                self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
                self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        if head_mode=="with_encoder":
            if angle_mode == "edge_delta":  
                self.pq_head = nn.Linear(hidden_dim, 1)    # PQ: predict Vmag only
                self.pv_head = nn.Linear(hidden_dim, 1)    # PV: predict Q only
                self.slack_head = nn.Linear(hidden_dim, 2) # Slack: predict P, Q
            else:
                self.pq_head = nn.Linear(hidden_dim, 2)      # for PQ nodes
                self.pv_head = nn.Linear(hidden_dim, 2)      # For PV nodes
                self.slack_head = nn.Linear(hidden_dim, 2)   # For Slack nodes

        # STSI 26.02.11: Edge embedding MLP: (h_src || h_dst || edge_attr) -> hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_features, hidden_dim),
            nn.LeakyReLU(),
        )

        # [STSI 250526]:edge angle head for edge_delta/both mode
        if angle_mode in ("edge_delta", "both"):  # [STSI 020626]:extended to support dual-head "both" mode
            self.edge_angle_pred = nn.Linear(hidden_dim, 1)

        # Bilinear PTDF parameter: h_edge^T W h_bus
        self.ptdf_W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        
        self._initialize_weights()

    def _make_conv(self, in_dim: int, out_dim: int, edge_features: int):
        if self.conv_type == "gatv2":
            return GATv2Conv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0, #self.dropout, # Dropout is handled separately in conv_mode, so set to 0 here to avoid double dropout when conv_mode="res_norm_drop"
            )
        elif self.conv_type == "transformer":  # [STSI120526] New conv_type option: dot-product attention, edge features in K+V, gated residual (beta=True)
            return TransformerConv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0,
                beta=not self.use_residual,  # [STSI 120526]:avoid double residual: external skip OR internal gated, not both
            )
        elif self.conv_type == "gcn":   # [STSI120526] DEPRECATED — no edge_attr support
            return GCNConv(in_dim, out_dim, add_self_loops=False)
        elif self.conv_type == "graphconv":   # [STSI120526] DEPRECATED — no edge_attr support
            return GraphConv(in_dim, out_dim)
        else:
            raise ValueError(f"Unknown conv_type: {self.conv_type}")

    def _apply_conv(self, conv, h, edge_index, edge_attr):
        """Apply a single conv layer, dispatching on conv type."""
        if isinstance(conv, (GATv2Conv, TransformerConv)):  # [STSI120526] Added TransformerConv to edge_attr dispatch
            return conv(h, edge_index, edge_attr)
        else:
            return conv(h, edge_index)

    def forward(self, data, return_embeddings=False):
        """
        Forward pass: embed -> GAT layers -> node heads + bilinear PTDF.

        Args:
            data: PyG Data/Batch object
            return_embeddings: if True, return (node_pred, h_nodes, h_edges)
                               for use in PTDF loss outside forward
        Returns:
            (node_pred [num_nodes, 4], ptdf_pred [num_edges, num_nodes])
            or (node_pred, h_nodes, h_edges) if return_embeddings=True
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Initial embedding
        h = self.node_embedding(x)

        # Conv layers — unified path with independent flags [STSI 120526]:replaced 4-way if/elif
        for i, conv in enumerate(self.convs):
            h_in = h
            h = self._apply_conv(conv, h, edge_index, edge_attr)
            if self.norm_type is not None:
                h = self.norms[i](h)
            h = self.activation_fn(h)
            if self.use_dropout:
                h = self.drop(h)
            if self.use_residual:
                h = h + h_in


        h_nodes = h  # Final node features after all GAT layers

        # Global pooling context injection [STSI 290526]: broadcast graph-level mean+max into per-node repr
        if self.use_global_pool:
            batch_idx = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(
                h_nodes.size(0), dtype=torch.long, device=h_nodes.device)
            h_mean = global_mean_pool(h_nodes, batch_idx)   # [num_graphs, hidden_dim]
            h_max = global_max_pool(h_nodes, batch_idx)     # [num_graphs, hidden_dim]
            h_global = self.global_proj(torch.cat([h_mean, h_max], dim=-1))  # [num_graphs, hidden_dim]
            h_nodes = h_nodes + h_global[batch_idx]  # additive residual injection


        # Node heads
        # Predictions are made using the output of the last graph convolution layer

        if self.head_mode=="standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]: node pred is [N,3]
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 3]
            else:
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                vang_pred = self.vang_pred(h_nodes)
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, vang_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 4]
        elif self.head_mode=="with_encoder":
            pq_mask = data.pq_mask.bool()
            pv_mask = data.pv_mask.bool()
            slack_mask = data.slack_mask.bool()
            if self.angle_mode == "edge_delta":  # [STSI 250526]: [N,3]=[Vmag,P,Q]
                node_pred = torch.zeros(h_nodes.size(0), 3, device=h_nodes.device)
                if pq_mask.any():
                    raw_vmag = self.pq_head(h_nodes[pq_mask]).squeeze(-1)  # Vmag
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        node_pred[pq_mask, 0] = raw_vmag + 1.0
                    else:
                        node_pred[pq_mask, 0] = raw_vmag
                if pv_mask.any():
                    node_pred[pv_mask, 2] = self.pv_head(h_nodes[pv_mask]).squeeze(-1)  # Q
                if slack_mask.any():
                    slack_out = self.slack_head(h_nodes[slack_mask])
                    node_pred[slack_mask, 1] = slack_out[:, 0]  # P
                    node_pred[slack_mask, 2] = slack_out[:, 1]  # Q
            else:
                node_pred = torch.zeros(h_nodes.size(0), 4, device=h_nodes.device)
                if pq_mask.any():
                    pq_out = self.pq_head(h_nodes[pq_mask])  # [N_pq, 2] = [Vmag, Vang]
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        pq_out = pq_out.clone()
                        pq_out[:, 0] = pq_out[:, 0] + 1.0
                    node_pred[pq_mask, 0:2] = pq_out
                if pv_mask.any():
                    pv_out=self.pv_head(h_nodes[pv_mask])  # Predict Vmag, Vang for PV nodes
                    node_pred[pv_mask,1]=pv_out[:,0]  # Vmag from PV head
                    node_pred[pv_mask,3]=pv_out[:,1]  # Q from PV head
                if slack_mask.any():
                    slack_out=self.slack_head(h_nodes[slack_mask])  # Predict Vmag, Vang for Slack nodes
                    node_pred[slack_mask, 2] = slack_out[:, 0]  # P from Slack head
                    node_pred[slack_mask, 3] = slack_out[:, 1]  # Q from Slack head

        # Edge embeddings
        row, col = data.edge_index
        h_src = h_nodes[row]   # [num_edges, hidden_dim]
        h_dst = h_nodes[col]   # [num_edges, hidden_dim]
        edge_input = torch.cat([h_src, h_dst, edge_attr], dim=-1)  # [num_edges, 2*hidden_dim + edge_features]
        h_edges = self.edge_mlp(edge_input)  # [num_edges, hidden_dim]

        # [STSI 250526]:edge angle prediction for edge_delta mode
        delta_theta_pred = None
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:dual-head produces Δθ in both modes
            # All forward edges (lines + transformers) — even indices
            h_edges_fwd = h_edges[::2]    # [E_all_fwd, hidden_dim] — even indices = forward edges
            delta_theta_pred = self.edge_angle_pred(h_edges_fwd).squeeze(-1)  # [E_fwd]

        if return_embeddings:
            return node_pred, h_nodes, h_edges, delta_theta_pred

        # Bilinear PTDF: H_W = h_edges @ W,  ptdf_pred = H_W @ h_nodes.T
        H_W = h_edges @ self.ptdf_W                   # [num_edges, hidden_dim]
        ptdf_pred = H_W @ h_nodes.T                   # [num_edges, num_nodes]
        return node_pred, ptdf_pred, delta_theta_pred

    def _initialize_weights(self, gain=1.0):
        """
        STSI 24.09.25: Enhanced weight initialization for the entire network.
        """
        # 1. Initialize node embedding layer (Added 24.09.25)
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)

        # 2. Initialize GAT convolution layers (Added 24.09.25)
        for conv in self.convs:
            if hasattr(conv, "lin_l") and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, "lin_r") and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, "lin_edge") and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
            # GAT layers have multiple linear transformations

        # 3. Initialize output layers
        _vmag_bias = 0.0 if self.vmag_mode == "residual" else 1.0  # [STSI 280526]
        if self.head_mode == "standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]no angle head, only Vmag, P, Q
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            else:
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.vang_pred, 0.0),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            for layer, bias_init in prediction_layers:
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.constant_(layer.bias, bias_init)
            # Zero-init vmag_pred weights so output starts exactly at bias [STSI 280526]
            nn.init.zeros_(self.vmag_pred.weight)
        elif self.head_mode == "with_encoder":
            for layer in (self.pq_head, self.pv_head, self.slack_head):
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.zeros_(layer.bias)
            # Vmag outputs (col 0 of pq_head) should start near 1.0 pu
            if self.angle_mode != "edge_delta":  # [STSI 250526]:only for 2-output pq_head
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)
            else:
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)  # single Vmag output
            nn.init.zeros_(self.pq_head.weight[0])  # Zero-init Vmag row [STSI 280526]

        # [STSI 250526]:initialize edge angle head  # [STSI 020626]:Fix — "both" also needs zero init
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:was == "edge_delta" only
            nn.init.zeros_(self.edge_angle_pred.weight)
            nn.init.zeros_(self.edge_angle_pred.bias)

        # [STSI 290526]: init global pooling projection
        if self.use_global_pool:
            nn.init.xavier_uniform_(self.global_proj.weight)


# [COPY of GNN_Powerflow_V2.7_Training::PhysicsConfig]
@dataclass
class PhysicsConfig:
    """
    Toggles for every physics guidance term.
    Designed to be swept as a hyperparameter in run_hparam_sweep.
    """
    w_phys: float = 0.0                     # static weight on physics loss (used in "fixed" mode)

    # Dynamic loss weighting [STSI 200526]
    loss_weight_mode: str = "fixed"         # "fixed" | "adaptive"
    fraction_physics: float = 0.1           # target: physics = 10% of MSE (adaptive only)
    fraction_ptdf: float = 0.1              # target: PTDF = 10% of MSE (adaptive only)
    max_w_phys: float = 100.0               # cap for adaptive physics weight
    max_w_ptdf: float = 100.0               # cap for adaptive PTDF weight

    # Variant selection
    physics_mode: str = "rich"            # reserved — "rich" only; "simple" removed 2026-05-21  # [STSI 210526]

    # Shared options
    use_power_balance: bool = True          # AC residual on P,Q
    use_angle_ref_penalty: bool = False     # slack angle mean^2

    # P/Q residual weights [STSI 210526]
    w_P: float = 0.5    # weight on P residual in physics loss (guides v_ang)
    w_Q: float = 0.5    # weight on Q residual in physics loss (guides v_mag)

    # Rich-mode options (full PV/slack handling, partial Q)
    use_q_partial_mode: bool = False        # exclude PV buses from Q residual
    
    # PTDF supervision
    use_ptdf_loss: bool = False    # DC PTDF head on/off
    weight_ptdf: float = 0.0      # weight applied to the PTDF loss term
    ptdf_loss_mode: str = "matrix" # "matrix", "flows", "mixed", or "learned_edge" (STSI 26.04.07)
    ptdf_alpha: float = 0.5       # mix ratio for mode="mixed" (matrix share)
    ptdf_branch_mode: str = "lines"  # branches in PTDF: "lines" or "all"
    ptdf_changepoint: int = None     # [STSI 290526] epoch offset (from warmup_epochs_ptdf) to switch matrix→flows in "stepwise" mode

    # ── Per-sub-loss flow fractions (replace w_flow hierarchy) ── [STSI 030626]
    fraction_dcf_local: float = 0.0      # adaptive target for DC-local flow loss (0=disabled)
    fraction_dcf_global: float = 0.0     # adaptive target for DC-global flow loss (0=disabled)
    fraction_acf_local: float = 0.0      # adaptive target for AC-local flow loss (0=disabled)
    fraction_acf_global: float = 0.0     # adaptive target for AC-global flow loss (0=disabled)
    max_w_flow: float = 100.0            # cap for all flow sub-loss adaptive weights
    flow_target: str = "pq"              # "p" (P only), "q" (Q only), or "pq" (both, AC only)

    # ── Tier 0: KVL loop-closure — own independent gate, no dependency on DC/AC flow schedule
    fraction_kvl: float = 0.0            # adaptive target for KVL loop-closure loss (0=disabled)
    max_w_kvl: float = 100.0             # cap for KVL adaptive weight (independent of max_w_flow)

    # ── Tier 1 DC-linear power balance (KCL) — [STSI 220926], research 2026-09-21
    enable_balance_dc: bool = False       # on/off switch; sized by fraction_dcf_local/_global (no new fraction)
    balance_dc_clamp_factor: float = 3.0  # [STSI 220926-b]: k in ±k*s_nom clamp on f_dc; <=0 disables clamp (Finding 2 mitigation, task 078)

    def label(self) -> str:
        # [STSI 200526] Distinguishes adaptive vs fixed mode in run labels
        if self.loss_weight_mode == "adaptive":
            parts = [f"adap-fp{self.fraction_physics}"]
            if self.use_ptdf_loss:
                parts.append(f"ft{self.fraction_ptdf}")
        else:
            parts = [f"w-{self.w_phys}"]
            if self.use_ptdf_loss:
                ptdf_part = f"PTDF-{self.ptdf_loss_mode}-pt{self.weight_ptdf}-{self.ptdf_branch_mode}"
                if self.ptdf_loss_mode == "mixed":
                    ptdf_part += f"-a{self.ptdf_alpha}"
                if self.ptdf_loss_mode == "stepwise" and self.ptdf_changepoint is not None:
                    ptdf_part += f"-cp{self.ptdf_changepoint}"
                parts.append(ptdf_part)
        if self.w_P != 0.5 or self.w_Q != 0.5:  # [STSI 210526]: non-default weights
            parts.append(f"wP{self.w_P}wQ{self.w_Q}")
        # [STSI 030626]:show active flow fractions in label
        _fl_parts = []
        if self.fraction_dcf_local > 0: _fl_parts.append(f"dcL{self.fraction_dcf_local}")
        if self.fraction_dcf_global > 0: _fl_parts.append(f"dcG{self.fraction_dcf_global}")
        if self.fraction_acf_local > 0: _fl_parts.append(f"acL{self.fraction_acf_local}")
        if self.fraction_acf_global > 0: _fl_parts.append(f"acG{self.fraction_acf_global}")
        if _fl_parts:
            parts.append("flow-" + "-".join(_fl_parts))
        if self.fraction_kvl > 0:
            parts.append(f"kvl{self.fraction_kvl}")
        if self.enable_balance_dc:  # [STSI 220926]
            parts.append("balDC")
            if self.balance_dc_clamp_factor != 3.0:  # [STSI 220926-b]: surface non-default clamp factor
                parts.append(f"clampK{self.balance_dc_clamp_factor}")
        return "_".join(parts)


# [COPY of GNN_Powerflow_V2.7_Training::compute_ptdf_matrix]
def compute_ptdf_matrix(network, ptdf_branch_mode="lines"):  # [STSI 260526]:added ptdf_branch_mode param 
    """
    Compute the Power Transfer Distribution Factor (PTDF) matrix for a PyPSA network.
    Returns a numpy array indexed by branches, columns by buses.

    Args:
        ptdf_branch_mode: "lines" returns [n_lines, n_buses],
                          "all" returns [n_lines + n_trafos, n_buses].
    """
    buses = list(network.buses.index)
    lines = list(network.lines.index)
    n_buses = len(buses)
    n_lines = len(lines)

    bus_to_idx = {b: i for i, b in enumerate(buses)}

    # Build susceptance matrix B and branch-bus incidence A
    B = np.zeros((n_buses, n_buses))
    n_trafos = len(network.transformers)  # [STSI 260526]:compute branch count
    n_branches = n_lines + n_trafos if ptdf_branch_mode == "all" else n_lines
    A = np.zeros((n_branches, n_buses))

    for i, line in enumerate(lines):
        b_val = 1.0 / network.lines.loc[line, "x"] if network.lines.loc[line, "x"] != 0 else 0.0
        from_idx = bus_to_idx[network.lines.loc[line, "bus0"]]
        to_idx = bus_to_idx[network.lines.loc[line, "bus1"]]
        B[from_idx, from_idx] += b_val
        B[to_idx, to_idx] += b_val
        B[from_idx, to_idx] -= b_val
        B[to_idx, from_idx] -= b_val
        A[i, from_idx] = b_val
        A[i, to_idx] = -b_val


    # Add transformer susceptance to B (transformers are series impedances in flat-pu)  # [STSI 230526]:fix missing trafo in B matrix
    for trafo_i, trafo in enumerate(network.transformers.index):
        row = network.transformers.loc[trafo]
        if "x_pu_eff" in row.index and pd.notna(row.get("x_pu_eff")) and float(row["x_pu_eff"]) != 0:
            x_val = float(row["x_pu_eff"])
        else:
            x_val = float(row["x"])
        if x_val == 0:
            continue
        b_val = 1.0 / x_val
        from_idx = bus_to_idx[row["bus0"]]
        to_idx = bus_to_idx[row["bus1"]]
        B[from_idx, from_idx] += b_val
        B[to_idx, to_idx] += b_val
        B[from_idx, to_idx] -= b_val
        B[to_idx, from_idx] -= b_val
        if ptdf_branch_mode == "all":  # [STSI 260526]:include trafo rows in incidence matrix A 
            A[n_lines + trafo_i, from_idx] = b_val
            A[n_lines + trafo_i, to_idx] = -b_val
    # Identify slack bus (first generator with control='Slack')
    slack_bus = None
    for gen in network.generators.index:
        if network.generators.loc[gen, "control"] == "Slack":
            slack_bus = network.generators.loc[gen, "bus"]
            break
    if slack_bus is None:
        slack_bus = buses[0]
    slack_idx = bus_to_idx[slack_bus]

    # Reduce B matrix (remove slack row/col)
    non_slack = [i for i in range(n_buses) if i != slack_idx]
    B_red = B[np.ix_(non_slack, non_slack)]
    A_red = A[:, non_slack]

    try:
        B_red_inv = np.linalg.inv(B_red)
    except np.linalg.LinAlgError:
        B_red_inv = np.linalg.pinv(B_red)

    PTDF_red = A_red @ B_red_inv  # (n_branches, n_buses-1)

    # Re-insert slack column as zeros
    PTDF = np.zeros((n_branches, n_buses))  # [STSI 260526]:use n_branches for mode=all
    non_slack_col = 0
    for j in range(n_buses):
        if j != slack_idx:
            PTDF[:, j] = PTDF_red[:, non_slack_col]
            non_slack_col += 1

    return PTDF


# [STSI 141026]:verbatim copies extracted programmatically (task 088); do not edit, originals are ground truth

In [ ]:
# NOTE: deliberate exception to DataGen>Training>Analysis precedence (user decision D13, 2026-10-14).


# PRECEDENCE EXCEPTION (D13 scope extension, user 2026-10-14): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::precompute_bfs_order]
def precompute_bfs_order(edge_index_fwd, slack_idx, num_nodes):  # [STSI 260526]:BFS traversal for theta reconstruction
    """
    Compute BFS traversal from slack for θ reconstruction from Δθ.
    
    Returns:
        bfs_edge_idx:   [N-1] indices into forward edges — edges in BFS tree order
        bfs_signs:      [N-1] float tensor — +1 or -1
        bfs_node_order: [N-1] node indices in BFS visit order (excludes slack)
        bfs_parent:     [N-1] parent node index for each BFS child
    
    Convention: θ_child = θ_parent - sign * Δθ_edge
        sign = +1 when edge is parent→child (forward direction = from→to)
        sign = -1 when edge is child→parent (traverse against forward direction)
    """
    # constructing the adjecency matrix for BFS traversal, with edge indices and signs for forward/backward direction
    # use manual rather than graph extaction because we need to track edge indices and signs for both forward and backward traversal, which is not directly provided by PyG's utilities
    adj = [[] for _ in range(num_nodes)]
    n_edges = edge_index_fwd.size(1)
    for e in range(n_edges):
        u = edge_index_fwd[0, e].item()
        v = edge_index_fwd[1, e].item()
        adj[u].append((v, e, +1.0))
        adj[v].append((u, e, -1.0))
    
    visited = [False] * num_nodes
    visited[slack_idx] = True
    queue = deque([slack_idx])
    
    bfs_edge_idx = []
    bfs_signs = []
    bfs_node_order = []
    bfs_parent = []
    
    while queue:
        node = queue.popleft()
        for neighbor, edge_e, sign in adj[node]:
            if not visited[neighbor]:
                visited[neighbor] = True
                bfs_edge_idx.append(edge_e)
                bfs_signs.append(sign)
                bfs_node_order.append(neighbor)
                bfs_parent.append(node)
                queue.append(neighbor)
    
    assert len(bfs_node_order) == num_nodes - 1, (
        f"BFS did not reach all nodes: visited {len(bfs_node_order)+1}/{num_nodes}"
    )
    
    return (
        torch.tensor(bfs_edge_idx, dtype=torch.long),
        torch.tensor(bfs_signs, dtype=torch.float32),
        torch.tensor(bfs_node_order, dtype=torch.long),
        torch.tensor(bfs_parent, dtype=torch.long),
    )


# PRECEDENCE EXCEPTION (D13 scope extension, user 2026-10-14): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::reconstruct_theta_from_delta]
def reconstruct_theta_from_delta(delta_theta_fwd, bfs_edge_idx, bfs_signs,  # [STSI 260526]:theta reconstruction from edge deltas
                                  bfs_node_order, bfs_parent, num_nodes):
    """
    Reconstruct absolute θ from edge Δθ predictions using pre-computed BFS order.
    Differentiable w.r.t. delta_theta_fwd (chain of additions).
    
    Args:
        delta_theta_fwd: [E_fwd] predicted Δθ per forward edge
        bfs_edge_idx:    [N-1] edge indices in BFS traversal order
        bfs_signs:       [N-1] signs (+1/-1)
        bfs_node_order:  [N-1] child node indices in BFS visit order
        bfs_parent:      [N-1] parent node index for each child
        num_nodes:       int, total nodes
    
    Returns:
        theta: [N] reconstructed absolute angles (θ_slack = 0)
    """
    theta = torch.zeros(num_nodes, device=delta_theta_fwd.device, dtype=delta_theta_fwd.dtype)
    
    for i in range(len(bfs_node_order)):
        child = bfs_node_order[i].item()
        parent_node = bfs_parent[i].item()
        e = bfs_edge_idx[i]
        s = bfs_signs[i]
        theta[child] = theta[parent_node] - s * delta_theta_fwd[e]
    
    return theta


# [COPY of GNN_Powerflow_V2.7_Analysis::reconstruct_theta_from_path_matrix]
def reconstruct_theta_from_path_matrix(delta_theta_fwd, bfs_path_matrix):  # [STSI 240726]:vectorized θ recon via path-incidence matrix
    """
    Reconstruct absolute θ from edge Δθ using precomputed path-incidence matrix.
    Replaces the sequential BFS loop with a single matmul: θ = P @ Δθ.
    P[i, e] encodes the sign of edge e on the spanning-tree path from slack to node i.
    θ_slack = 0 (slack row of P is all zeros).

    For meshed networks, the result is BFS-tree-dependent (spanning-tree choice matters).
    delta_theta_mae is the tree-independent primary angle metric for edge_delta models.

    Args:
        delta_theta_fwd:  [E_fwd] tensor of predicted Δθ per forward edge
        bfs_path_matrix:  [N, E_fwd] precomputed path-incidence matrix (from Data)
    Returns:
        theta: [N] reconstructed absolute angles (θ_slack = 0)
    """
    return bfs_path_matrix.to(delta_theta_fwd.device) @ delta_theta_fwd


# PRECEDENCE EXCEPTION (D13): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::PowerFlowDataset]
class PowerFlowDataset(Dataset):
    """
    One data object per (network, snapshot) pair.

    The graph topology reflects the physical network topology:
    - buses → GNN nodes
    - lines + transformers → GNN edges (bidirectional)

    Each graph carries:
      x         : [n_buses, 7 or 8] node features (bus type flags, known P/Q/V/Vang, optional p_nom_share)
      y         : [n_buses, 4]       targets (vmag, vang, p, q)
      edge_index: [2, n_edges]       bidirectional
      edge_attr : [n_edges, 6]       edge features (r, x, b/2, tap, g_series, b_series)
      masks     : slack_mask, pv_mask, pq_mask
      y_line_p  : [n_fwd_lines]      forward-direction real power per supervised edge (PTDF supervision)
      ptdf_edge_row_idx : [n_edges]  PTDF row index for each edge (-1 for non-supervised/reverse)  # STSI 26.04.07
      network_idx: int
    """

    def __init__(
        self,
        networks: list,
        use_edge_features: bool = True,
        use_pnet_balance: bool = True,   # [STSI100526] kept for backfill compat (ignored internally)
        use_pnom_share: bool = False,     # [STSI100526] col7: p_nom_share (was col7=p_bal + col8=pnom_share before 100526)
        transform=None,
        pre_transform=None,
        ptdf_branch_mode: str = "lines",  # "lines" or "all" — controls PTDF supervision scope
        angle_mode: str = "node",          # [STSI 250526] "node" (default) or "edge_delta"
    ):
        super().__init__(root=None, transform=transform, pre_transform=pre_transform)
        self.networks          = networks
        self.use_edge_features = use_edge_features
        self.use_pnom_share    = use_pnom_share
        self.ptdf_branch_mode  = ptdf_branch_mode #STSI260402: pass ptdf_branch_mode to _create_graph_data for edge feature control
        self.angle_mode        = angle_mode  # [STSI 250526] "node" or "edge_delta"

        # Build flat index: list of (network_idx, timestep_idx)
        self._index = []
        for net_idx, net in enumerate(networks):
            for t_idx in range(len(net.snapshots)):
                self._index.append((net_idx, t_idx))

    def len(self):
        return len(self._index)

    def get(self, idx):
        net_idx, t_idx = self._index[idx]
        network = self.networks[net_idx]
        data = self._create_graph_data(network, t_idx, net_idx)
        return data

    def _create_graph_data(self, network: pypsa.Network, t_idx: int, net_idx: int) -> Data:
        buses     = list(network.buses.index)
        n_buses   = len(buses)
        bus_to_i  = {b: i for i, b in enumerate(buses)}
        snapshot  = network.snapshots[t_idx]

        # ── Bus type flags ────────────────────────────────────────────────────
        is_slack = torch.zeros(n_buses, dtype=torch.float)
        is_pv    = torch.zeros(n_buses, dtype=torch.float)
        is_pq    = torch.zeros(n_buses, dtype=torch.float)

        slack_mask = torch.zeros(n_buses, dtype=torch.bool)
        pv_mask    = torch.zeros(n_buses, dtype=torch.bool)
        pq_mask    = torch.zeros(n_buses, dtype=torch.bool)

        for gen_name, gen in network.generators.iterrows():
            bus_name = gen["bus"]
            if bus_name not in bus_to_i:
                continue
            i = bus_to_i[bus_name]
            ctrl = gen["control"]
            if ctrl == "Slack":
                is_slack[i] = 1.0
                slack_mask[i] = True
            elif ctrl == "PV":
                is_pv[i] = 1.0
                pv_mask[i] = True

        # Buses with no generator → PQ
        for i in range(n_buses):
            if not slack_mask[i] and not pv_mask[i]:
                is_pq[i]   = 1.0
                pq_mask[i] = True

        # ── Bus injections (P, Q) from PyPSA post-solve ───────────────────────
        # STSI260404: BUGFIX — use .loc[snapshot, buses] to force column order
        # to match network.buses.index.  Without this, buses_t columns may be
        # in lexicographic order (e.g. "Bus 10" before "Bus 2"), mis-aligning
        # P/Q/Vang with bus-type masks and the Y-matrix built from buses.index.
        p_bus = torch.tensor(
            network.buses_t.p.loc[snapshot, buses].values, dtype=torch.float
        )
        q_bus = torch.tensor(
            network.buses_t.q.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Voltages ─────────────────────────────────────────────────────────────
        v_mag = torch.tensor(
            network.buses_t.v_mag_pu.loc[snapshot, buses].values, dtype=torch.float
        )
        v_ang = torch.tensor(
            network.buses_t.v_ang.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Input features: GNN receives known values, zeros where unknown ────
        # Known before solve: bus type flags, P_inj for PQ+PV, Q_inj for PQ,
        #                     Vmag for PV+Slack, Vang for Slack (=0)
        #
        # x layout: [is_slack(0), is_PV(1), is_PQ(2), P(3), Q(4), Vmag(5), Vang(6)]
        #
        # NOTE: Slack P input x[:,3]=0 is a known issue — the model sees no
        # varying signal for slack P.  # [STSI100526] removed p_net_balance (was here)
        x_p    = p_bus.clone()
        x_q    = q_bus.clone()
        x_vmag = v_mag.clone()
        x_vang = v_ang.clone()

        # Mask unknowns to zero in input (prevents trivial leakage)
        x_p[slack_mask]    = 0.0   # Slack P unknown (to be predicted)
        x_q[slack_mask]    = 0.0   # Slack Q unknown
        x_q[pv_mask]       = 0.0   # PV Q unknown
        x_vmag[pq_mask]    = 0.0   # PQ Vmag unknown
        x_vang[pq_mask]    = 0.0   # PQ Vang unknown
        x_vang[pv_mask]    = 0.0   # PV Vang unknown

        x = torch.stack([is_slack, is_pv, is_pq, x_p, x_q, x_vmag, x_vang], dim=1)

        if self.use_pnom_share:  # [STSI100526] removed p_bal_col (redundant); GNN computes load sum internally
            # col7: static p_nom share per gen bus (p_nom_i / Σp_nom)
            # Tells the model each generator's relative capacity for slack distribution.
            p_nom_share_col = torch.zeros(n_buses, 1)
            p_noms = network.generators["p_nom"].values.astype(float)
            total_p_nom = float(p_noms.sum()) if p_noms.sum() > 0 else 1.0
            for gen_name, gen in network.generators.iterrows():
                b = gen["bus"]
                if b in bus_to_i:
                    p_nom_share_col[bus_to_i[b], 0] += gen["p_nom"] / total_p_nom
            x = torch.cat([x, p_nom_share_col], dim=1)

        # ── Target: full post-solve state for all buses ──────────────────────────
        y = torch.stack([v_mag, v_ang, p_bus, q_bus], dim=1)

        # ── Edge construction (lines + transformers) ──────────────────────────────
        edge_index_list = []
        edge_attr_list  = []
        forward_edge_mask = []   # True for first direction (not reverse duplicate)
        dc_flow_mask = []        # [STSI 020626]:True only for forward LINE edges (not transformers) — for DC flow loss
        fwd_line_idx_list  = []   # one entry per forward supervised edge
        ptdf_edge_row_idx_list = []  # STSI 26.04.07: PTDF row index per edge for learned_edge mode

        branches = []
        #STSI260402: control which branches are included in PTDF supervision via self.ptdf_branch_mode
        # lines always included
        for line_name, line in network.lines.iterrows():
            if line["bus0"] in bus_to_i and line["bus1"] in bus_to_i:
                branches.append(
                    (
                        line.name,  # name (not used in PTDF but useful for debugging)
                        line["bus0"], line["bus1"],
                        line["r"], line["x"],
                        line.get("b", 0.0), 1.0, "line"
                    )
                )

        # STSI260404: BUGFIX — transformers must ALWAYS be added as graph edges
        # for GNN message passing, regardless of ptdf_branch_mode.
        # ptdf_branch_mode only controls which branches are supervised by PTDF
        # loss (via forward_edge_mask / fwd_line_idx_list), NOT graph topology.
        # Previously gated behind `if self.ptdf_branch_mode == "all":` which
        # silently severed the graph when mode was "lines" (the default).
        for tr_name, tr in network.transformers.iterrows():
            if tr["bus0"] in bus_to_i and tr["bus1"] in bus_to_i:
                tap = tr.get("tap_ratio", 1.0)
                tap = tap if not pd.isna(tap) else 1.0
                branches.append(
                    (
                        tr.name,  # name (not used in PTDF but useful for debugging)
                        tr["bus0"], tr["bus1"],
                        tr["r"], tr["x"],
                        tr.get("b", 0.0), float(tap), "trafo"
                    )
                )


        for branch_idx, (name, b0, b1, r, x_val, b_val, tap, btype) in enumerate(branches):
            i0, i1 = bus_to_i[b0], bus_to_i[b1]
            z      = complex(r, x_val)
            y_ser  = (1.0 / z) if abs(z) > 1e-12 else 0.0
            g_ser, b_ser = y_ser.real, y_ser.imag

            if self.use_edge_features:
                attr = [r, x_val, b_val / 2.0, tap, g_ser, b_ser]
            else:
                attr = [1.0]

            # Forward edge
            edge_index_list.append([i0, i1])
            edge_attr_list.append(attr)

            # PTDF supervision: only lines (or all branches) get True in forward_edge_mask
            is_supervised = (btype == "line") if self.ptdf_branch_mode == "lines" else True
            forward_edge_mask.append(is_supervised)
            dc_flow_mask.append(btype == "line")  # [STSI 020626]:DC mask excludes transformers
            if is_supervised:
                fwd_line_idx_list.append(branch_idx)
                ptdf_edge_row_idx_list.append(branch_idx)  # STSI 26.04.07: supervised forward edge
            else:
                ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: unsupervised forward edge

            # Reverse edge
            edge_index_list.append([i1, i0])
            edge_attr_list.append(attr)
            forward_edge_mask.append(False)
            dc_flow_mask.append(False)  # [STSI 020626]:reverse edge always False
            ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: reverse edge

        if edge_index_list:
            edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float)
            fwd_mask   = torch.tensor(forward_edge_mask, dtype=torch.bool)
            dc_fwd_mask = torch.tensor(dc_flow_mask, dtype=torch.bool)  # [STSI 020626]:DC-only line mask
            line_idx   = torch.tensor(fwd_line_idx_list, dtype=torch.long)
            ptdf_edge_row_idx = torch.tensor(ptdf_edge_row_idx_list, dtype=torch.long)  # STSI 26.04.07
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr  = torch.zeros((0, len(attr) if branches else 1), dtype=torch.float)
            fwd_mask   = torch.zeros(0, dtype=torch.bool)
            dc_fwd_mask = torch.zeros(0, dtype=torch.bool)  # [STSI 020626]:DC-only line mask (empty)
            line_idx   = torch.zeros(0, dtype=torch.long)
            ptdf_edge_row_idx = torch.zeros(0, dtype=torch.long)  # STSI 26.04.07


        try:
            ptdf_matrix = compute_ptdf_matrix(network, ptdf_branch_mode=self.ptdf_branch_mode)  # [STSI 260526]:pass branch mode 
            y_ptdf = torch.tensor(ptdf_matrix, dtype=torch.float)
        except Exception:
            n_lines_fb = len(network.lines)
            n_branches_fb = n_lines_fb + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines_fb
            y_ptdf = torch.zeros((n_branches_fb, n_buses), dtype=torch.float)  # [STSI 260526]:use branch count in fallback 

        # STSI260402 Assertions — verify PTDF supervision scope and consistency
        n_lines = len(network.lines)
        if self.ptdf_branch_mode == "lines":
            assert int(fwd_mask.sum()) == n_lines, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines {n_lines}"
        else:  # ptdf_branch_mode == "all": lines + transformers
            n_trafos = len(network.transformers)
            assert int(fwd_mask.sum()) == n_lines + n_trafos, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines+n_trafos {n_lines + n_trafos}"
        expected_ptdf_rows = n_lines + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines  # [STSI 260526]: expected PTDF rows based on supervised branches
        assert y_ptdf.shape[0] == expected_ptdf_rows, \
            f"y_ptdf rows {y_ptdf.shape[0]} != expected {expected_ptdf_rows} (mode={self.ptdf_branch_mode})"

        # ── Line active power flows ───────────────────────────────────────────
        if not network.lines_t.p0.empty:
            y_line_p = torch.tensor(
                network.lines_t.p0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_p = torch.zeros(len(network.lines), dtype=torch.float)

        # [STSI 260526]: Append transformer flows when supervising all branches 
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.p0.empty:
                y_trafo_p = torch.tensor(
                    network.transformers_t.p0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_p = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_p = torch.cat([y_line_p, y_trafo_p])

        # ── Q line flows (for AC per-edge flow loss) ──── [STSI 020626]
        if not network.lines_t.q0.empty:
            y_line_q = torch.tensor(
                network.lines_t.q0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_q = torch.zeros(len(network.lines), dtype=torch.float)

        # Append transformer Q flows when supervising all branches
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.q0.empty:
                y_trafo_q = torch.tensor(
                    network.transformers_t.q0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_q = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_q = torch.cat([y_line_q, y_trafo_q])


        # [STSI 260526]: Precompute diagonal admittance for edge-local physics 
        # g_diag[i] = Σ_j g_series_ij ; b_diag[i] = Σ_j (b_series_ij + b_half_ij)
        if edge_index.size(1) > 0 and edge_attr.size(0) > 0:
            from torch_geometric.utils import scatter
            _src_nodes = edge_index[0]  # [2E] source node indices
            _g_ser_all = edge_attr[:, 4]  # [2E] series conductance
            _b_ser_all = edge_attr[:, 5]  # [2E] series susceptance
            _b_half_all = edge_attr[:, 2]  # [2E] line charging b/2 per end
            # STSI 220926 (task 065, Bug B fix): forward-edge series scaled by 1/tap^2 so the
            # edge-local diagonal matches compute_admittance_matrix's Y_ii tap handling. tap=1 ->
            # no-op (regression-safe).
            _tap_all = edge_attr[:, 3].clamp(min=1e-6)
            _g_ser_diag = _g_ser_all.clone()
            _b_ser_diag = _b_ser_all.clone()
            _g_ser_diag[fwd_mask] = _g_ser_diag[fwd_mask] / (_tap_all[fwd_mask] ** 2)
            _b_ser_diag[fwd_mask] = _b_ser_diag[fwd_mask] / (_tap_all[fwd_mask] ** 2)
            g_diag = scatter(_g_ser_diag, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
            b_diag = scatter(_b_ser_diag + _b_half_all, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
            # STSI 220926 (task 065, Bug A fix): bus shunt impedances (bus-to-ground) -- diagonal
            # only add so edge-local physics matches PyPSA at shunt buses.
            if len(network.shunt_impedances) > 0:
                _bidx = {b: i for i, b in enumerate(network.buses.index)}
                for _sh in network.shunt_impedances.index:
                    _sr = network.shunt_impedances.loc[_sh]
                    _i = _bidx.get(_sr['bus'])
                    if _i is not None:
                        g_diag[_i] += float(_sr.get('g', 0.0) or 0.0)
                        b_diag[_i] += float(_sr.get('b', 0.0) or 0.0)
        else:
            g_diag = torch.zeros(n_buses, dtype=torch.float)
            b_diag = torch.zeros(n_buses, dtype=torch.float)

        # ── Assemble Data object ──────────────────────────────────────────────
        data = Data(
            x          = x,
            y          = y,
            edge_index = edge_index,
            edge_attr  = edge_attr,
        )

        # Bus-type masks — node-level bool tensors (critical for collate + loss)
        data.slack_mask = slack_mask   # [n_nodes] bool
        data.pv_mask    = pv_mask      # [n_nodes] bool
        data.pq_mask    = pq_mask      # [n_nodes] bool

        # Variable-size targets (handled specially in collate_with_ptdf)
        data.y_ptdf      = y_ptdf
        data.y_line_p    = y_line_p
        data.y_line_q    = y_line_q  # [STSI 020626]: reactive flow target for AC per-edge loss
        data.forward_edge_mask = fwd_mask
        data.dc_flow_mask = dc_fwd_mask  # [STSI 020626]:lines-only mask for DC flow loss (excludes transformers)
        data.g_diag = g_diag  # [STSI 260526]: diagonal conductance for edge-local physics
        data.b_diag = b_diag  # [STSI 260526]: diagonal susceptance for edge-local physics
        data.ptdf_line_index = line_idx
        data.ptdf_edge_row_idx = ptdf_edge_row_idx  # STSI 26.04.07: per-edge PTDF row index for learned_edge mode

        # Graph-level metadata
        data.network_idx    = torch.tensor([net_idx], dtype=torch.long)

        # ── Δθ target and BFS metadata (for angle_mode="edge_delta" or "both") ── [STSI 250526]
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:store Δθ targets for both modes
            # All forward edges (lines + transformers) for full BFS connectivity
            all_fwd_mask = torch.zeros(edge_index.size(1), dtype=torch.bool)
            all_fwd_mask[::2] = True  # even indices = forward edges
            edge_index_fwd = edge_index[:, all_fwd_mask]
            
            # Compute Δθ target: θ_from - θ_to for each forward edge
            theta_true = y[:, 1]  # absolute angles from power flow solution
            y_delta_theta = theta_true[edge_index_fwd[0]] - theta_true[edge_index_fwd[1]]
            data.y_delta_theta = y_delta_theta  # [E_fwd]
            
            # Precompute BFS order from slack
            slack_idx_val = int(slack_mask.nonzero(as_tuple=True)[0][0])
            bfs_edge_idx, bfs_signs, bfs_node_order, bfs_parent = precompute_bfs_order(
                edge_index_fwd, slack_idx_val, n_buses
            )

            # [STSI 240726]: Build dense path-incidence matrix P [N × E_fwd] for fast θ reconstruction
            # P[i, e] is non-zero for every edge e on the spanning-tree path from slack to node i.
            # P[child] = P[parent] (inherit full path) then P[child, edge] = -sign for the new edge.
            # θ = P @ delta_theta_fwd  (slack row is all zeros → θ_slack = 0)
            # Build in BFS order so parent rows are complete before child rows are computed.
            _e_fwd = edge_index_fwd.size(1)  # number of forward edges
            bfs_path_matrix = torch.zeros(n_buses, _e_fwd, dtype=torch.float32)
            for _bi in range(len(bfs_node_order)):  # BFS order guarantees parents precede children
                _child  = bfs_node_order[_bi].item()
                _parent = bfs_parent[_bi].item()
                _edge   = bfs_edge_idx[_bi].item()
                _sign   = bfs_signs[_bi].item()
                bfs_path_matrix[_child] = bfs_path_matrix[_parent].clone()  # inherit parent path
                bfs_path_matrix[_child, _edge] = -_sign  # add direct edge contribution
            # Note: for meshed networks, vang reconstructed via this BFS tree is tree-dependent
            # (spanning-tree choice matters). delta_theta_mae is tree-independent.
            data.bfs_path_matrix = bfs_path_matrix  # [N, E_fwd]
            data.bfs_edge_idx = bfs_edge_idx
            data.bfs_signs = bfs_signs
            data.bfs_node_order = bfs_node_order
            data.bfs_parent = bfs_parent

        return data


# PRECEDENCE EXCEPTION (D13): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::collate_with_ptdf]
def collate_with_ptdf(batch):
    """
    Custom collate function for variable-size PTDF matrices and per-graph attributes.
    
    FIX 2026-03-25: Explicitly propagate slack_mask, pv_mask, pq_mask so that
    _masked_mse_loss and physics_informed_loss_batch receive valid mask tensors.
    Without this, all masked losses silently return 0.

    """
    # ── Variable-size lists: remove before PyG batching ──────────────────────
    y_ptdf_list   = [data.y_ptdf   for data in batch]
    y_line_p_list = [data.y_line_p for data in batch]
    y_line_q_list = [data.y_line_q for data in batch]  # [STSI 020626]
    ptdf_line_index_list = [data.ptdf_line_index for data in batch]


    # [STSI 250526]:also remove BFS metadata before PyG batching (variable-size per graph)
    bfs_keys = ['bfs_edge_idx', 'bfs_signs', 'bfs_node_order', 'bfs_parent', 'y_delta_theta', 'bfs_path_matrix']  # [STSI 240726]:add bfs_path_matrix for vectorized θ recon
    bfs_data = {k: [] for k in bfs_keys}
    for data in batch:
        for k in bfs_keys:
            if hasattr(data, k):
                bfs_data[k].append(getattr(data, k))
                delattr(data, k)

    for data in batch:
        del data.y_ptdf
        del data.y_line_p #STSI260402: added deletion of y_line_p to prevent PyG from trying to batch it and causing errors due to variable sizes
        del data.y_line_q  # [STSI 020626]
        del data.ptdf_line_index
    # ── PyG auto-batches all node-level tensors (x, y, edge_index, etc.) ─────
    # slack_mask / pv_mask / pq_mask are node-level bool tensors stored on each
    # Data object by PowerFlowDataset — Batch.from_data_list will concat them
    # along the node dimension automatically IF they are present.
    # ptdf_edge_row_idx is edge-level [n_edges] long — also auto-concatenated (STSI 26.04.07)
    batched = Batch.from_data_list(batch)

    # ── Restore variable-size list attributes ────────────────────────────────
    batched.y_ptdf_list   = y_ptdf_list
    batched.y_line_p_list = y_line_p_list
    batched.y_line_q_list = y_line_q_list  # [STSI 020626]
    batched.ptdf_line_index_list = ptdf_line_index_list

    # [STSI 250526]:restore BFS metadata as lists (for angle_mode="edge_delta")
    for k, v_list in bfs_data.items():
        if v_list:
            setattr(batched, k, v_list)

    # Restore on originals for re-use in dataset
    for i, (data, y_ptdf, y_line_p, y_line_q, ptdf_idx) in enumerate(zip(batch, y_ptdf_list, y_line_p_list, y_line_q_list, ptdf_line_index_list)):  # [STSI 020626]:added y_line_q
        data.y_ptdf   = y_ptdf
        data.y_line_p = y_line_p
        data.y_line_q = y_line_q  # [STSI 020626]
        data.ptdf_line_index = ptdf_idx
        # [STSI 250526]:restore BFS attrs on originals
        for k in bfs_keys:
            if bfs_data[k]:
                setattr(data, k, bfs_data[k][i])

    # ── Defensive check: abort loudly if masks are missing ───────────────────
    for mask_attr in ("slack_mask", "pv_mask", "pq_mask"):
        if not hasattr(batched, mask_attr):
            raise RuntimeError(
                f"collate_with_ptdf: batched object missing '{mask_attr}'. "
                f"Ensure PowerFlowDataset stores data.{mask_attr} as a node-level "
                f"bool tensor (shape [n_nodes]) on every Data object."
            )

    return batched


# PRECEDENCE EXCEPTION (D13 scope extension, user 2026-10-14): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::calculate_line_flows]
def calculate_line_flows(network, vmag, vang, t_idx=None):
    """
    Calculate AC line flows from predicted voltages.

    Args:
        network: PyPSA network object
        vmag:    Voltage magnitudes [num_buses], numpy array, p.u.
        vang:    Voltage angles [num_buses], numpy array, radians
        t_idx:   Snapshot index (used for nominal values only)
    Returns:
        dict with keys p0, p1, q0, q1 - each a numpy array of length num_lines (p.u.)
    """
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)

    all_buses = list(network.buses.index)
    # BUGFIX: use bus_to_idx dict instead of string parsing (topology-safe)
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(all_buses)}

    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus = network.lines.loc[line, "bus1"]

        # BUGFIX: use dict lookup instead of int(bus.split('-')[1]) - 1
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]

        r = network.lines.loc[line, "r"]
        x = network.lines.loc[line, "x"]
        b = network.lines.loc[line, "b"] if "b" in network.lines.columns else 0.0

        # Line impedance and admittance
        z = complex(r, x)
        y_series = 1.0 / z if abs(z) > 1e-12 else 0.0
        y_shunt = complex(0, b / 2.0)

        # Complex voltages at from/to buses
        V_from = vmag[from_idx] * np.exp(1j * vang[from_idx])
        V_to = vmag[to_idx] * np.exp(1j * vang[to_idx])

        # Line current from sending end
        I_from = (V_from - V_to) * y_series + V_from * y_shunt
        I_to = (V_to - V_from) * y_series + V_to * y_shunt

        S_from = V_from * np.conj(I_from)
        S_to = V_to * np.conj(I_to)

        s_nom = network.lines.loc[line, "s_nom"]
        # Convert from p.u. to MW/MVAr using system base
        #base_mva = getattr(network, "sn_mva", 100.0)

        p0[i] = S_from.real #* base_mva # removed the base_mva scaling to keep flows in p.u. for better numerical stability and consistency with the rest of the model's inputs/outputs
        q0[i] = S_from.imag #* base_mva
        p1[i] = S_to.real #* base_mva  # STSI 220926 (task 065, sign fix): was -S_to.real; PyPSA p1 convention is NOT negated
        q1[i] = S_to.imag #* base_mva  # STSI 220926 (task 065, sign fix): was -S_to.imag

    return {"p0": p0, "p1": p1, "q0": q0, "q1": q1}


# PRECEDENCE EXCEPTION (D13 scope extension, user 2026-10-14): by-rule origin GNN_Powerflow_V2.7_Training
# [COPY of GNN_Powerflow_V2.7_Analysis::calculate_line_flows_from_delta_theta]
def calculate_line_flows_from_delta_theta(  # [STSI 270526]: task 028 — direct Δθ line flows (copied from Training V2.6.2)
    delta_theta,     # [E_fwd] numpy — Δθ per forward edge (θ_from - θ_to)
    vmag,            # [N] numpy — voltage magnitudes
    edge_index_fwd,  # [2, E_fwd] numpy — forward edge source/target
    edge_attr_fwd,   # [E_fwd, 6] numpy — [r, x, b_half, tap, g_ser, b_ser]
) -> dict:
    """
    Compute line flows directly from Δθ without θ reconstruction.
    Uses π-model: P_from = V_from²×(g_s+g_sh) - V_from×V_to×(g_s×cos(Δθ) + b_s×sin(Δθ))
    Returns dict with keys: 'p0', 'q0', 'p1', 'q1' (from/to in per-unit).
    """
    from_bus = edge_index_fwd[0]  # [E_fwd]
    to_bus = edge_index_fwd[1]    # [E_fwd]
    v_from = vmag[from_bus]
    v_to = vmag[to_bus]

    r = edge_attr_fwd[:, 0]
    x_imp = edge_attr_fwd[:, 1]
    b_half = edge_attr_fwd[:, 2]
    tap = edge_attr_fwd[:, 3]

    # Series admittance
    z = r + 1j * x_imp
    y_s = 1.0 / z
    g_s = y_s.real
    b_s = y_s.imag

    # Handle tap ratio for transformers
    tap_nz = np.where(np.abs(tap) > 1e-12, tap, 1.0)
    # Tap-adjusted: effective from-end voltage = V_from / tap
    v_from_eff = v_from / tap_nz

    # Shunt susceptance per end
    b_sh = b_half

    cos_dt = np.cos(delta_theta)
    sin_dt = np.sin(delta_theta)

    # From-end power (π-model with tap on from side)
    p0 = v_from_eff**2 * g_s - v_from_eff * v_to * (g_s * cos_dt + b_s * sin_dt)
    q0 = -v_from_eff**2 * (b_s + b_sh) - v_from_eff * v_to * (g_s * sin_dt - b_s * cos_dt)

    # To-end power (Δθ reversed; PyPSA convention S1 = V1·conj(I1) = power into the branch at bus1)  # [STSI 191026]:task 096 F7 - outer negation removed
    p1 = v_to**2 * g_s - v_to * v_from_eff * (g_s * cos_dt - b_s * sin_dt)
    q1 = -v_to**2 * (b_s + b_sh) + v_to * v_from_eff * (g_s * sin_dt + b_s * cos_dt)

    return {'p0': p0, 'q0': q0, 'p1': p1, 'q1': q1}


# [COPY of GNN_Powerflow_V2.7_Analysis::_augment_edge_attr_learned]
def _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset=0, device=None):
    """Augment data.edge_attr with learned PTDF rows for learned_edge models.

    Shared helper used by evaluate_model, compare_pf_conventional_vs_gnn,
    predict_network_results_with_masks, and evaluate_gnn_on_test_set.

    Args:
        data:          single PyG Data object (already moved to device)
        ptdf_params:   list[nn.Parameter] or None
        max_buses:     int, PTDF parameter width
        net_idx:       int, network index within the dataset
        ptdf_offset:   int, added to net_idx to get global ptdf_params index
        device:        torch device (inferred from data.edge_attr if None)

    Returns:
        data with edge_attr augmented in-place. No-op if ptdf_params is None.
    """
    if ptdf_params is None or max_buses <= 0:
        return data
    if device is None:
        device = data.edge_attr.device
    row_idx = data.ptdf_edge_row_idx  # [E] line index or -1
    ptdf_rows = torch.zeros(data.edge_attr.size(0), max_buses, device=device)
    valid = (row_idx >= 0) & (row_idx < ptdf_params[net_idx + ptdf_offset].size(0))
    if valid.any():
        ptdf_rows[valid] = ptdf_params[net_idx + ptdf_offset][row_idx[valid]].to(device)
    data.edge_attr = torch.cat([data.edge_attr, ptdf_rows], dim=-1)
    return data


# [COPY of GNN_Powerflow_V2.7_Analysis::predict_network_results_with_masks]
def predict_network_results_with_masks(
    model, network, use_edge_features=True, debug=False, use_pnom_share=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None, max_buses=0, ptdf_offset=0,
    collect_delta_theta=False,  # [STSI 010626]:optionally collect edge-level delta_theta pred/true
):
    """
    Run GNN prediction on a network and return bus/line results with prediction masks.

    DISTSLACK: prediction masks now also flag distributed slack buses correctly.
    Distributed slack buses predict P and Q (same as single slack).
    STSI 26.04.07: Augments edge_attr when ptdf_params is provided (learned_edge).

    Args:
        model:   Trained PowerFlowGNN model
        network: PyPSA network object (must have solved power flow for feature extraction)
    Returns:
        bus_results, line_results, prediction_masks
    """
    _angle_mode = getattr(model, "angle_mode", "node")
    dataset = PowerFlowDataset(
        [network],
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
        angle_mode=_angle_mode,  # [STSI 260526]:pass angle_mode
        ptdf_branch_mode="all" if _angle_mode in ("edge_delta", "both") else "lines",  # [STSI 260526]:edge_delta needs all branches for BFS connectivity
    )
    snapshots = network.snapshots
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product(
            [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
        ),
        dtype=float,
    )
    # Prediction masks: which properties were predicted (not taken from input)
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    model.eval()
    _dt_pred_rows, _dt_true_rows = [], []  # [STSI 010626]:delta_theta accumulators
    with torch.no_grad():
        for t_idx, t in enumerate(snapshots):
            data = dataset[t_idx]

            # STSI 26.04.07: augment edge_attr for learned_edge models
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx=0, ptdf_offset=ptdf_offset)

            # Fallback: zero-pad if model expects wider edge_attr (old runs without stored ptdf_params)
            if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
                _expected = model.convs[0].lin_edge.in_channels
                if data.edge_attr.size(1) < _expected:
                    data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)

            # [STSI 260526]:handle 3-tuple return and edge_delta theta reconstruction
            angle_mode = getattr(model, "angle_mode", "node")
            out = model(data)
            node_pred = out[0]
            delta_theta_pred = out[2] if len(out) > 2 else None
            if collect_delta_theta and delta_theta_pred is not None:  # [STSI 010626]:accumulate edge delta_theta
                _dt_pred_rows.append(delta_theta_pred.cpu().numpy())
                if hasattr(data, "y_delta_theta") and data.y_delta_theta is not None:
                    _dt_true_rows.append(data.y_delta_theta.cpu().numpy())

            # θ reconstruction for edge_delta mode [STSI 260526]
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                theta_recon = reconstruct_theta_from_delta(
                    delta_theta_pred,
                    data.bfs_edge_idx,
                    data.bfs_signs,
                    data.bfs_node_order,
                    data.bfs_parent,
                    data.x.size(0),
                )
                node_pred_full = torch.zeros(data.x.size(0), 4, device=node_pred.device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = theta_recon       # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P (col1 in 3-col)
                node_pred_full[:, 3] = node_pred[:, 2]  # Q (col2 in 3-col)
                node_pred = node_pred_full

            pred_vmag = node_pred[:, 0]
            pred_vang = node_pred[:, 1]
            pred_p    = node_pred[:, 2]
            pred_q    = node_pred[:, 3]

            for i, bus in enumerate(buses):
                bus_type = "PQ"  # default
                connected_gens = network.generators[network.generators.bus == bus]
                if len(connected_gens) > 0:
                    ctrl = connected_gens.iloc[0]["control"]
                    if ctrl in ("Slack", "PV"):
                        bus_type = ctrl

                if bus_type == "PQ":
                    # For PQ buses: predict vmag, vang
                    pred_vmag_i = pred_vmag[i]
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]   # Known
                    pred_q_i    = network.buses_t.q.loc[t, bus]   # Known
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = False
                    prediction_masks["V"].loc[t, bus]     = True
                    prediction_masks["Angle"].loc[t, bus] = True

                elif bus_type == "PV":
                    # For PV buses: known vmag, predict vang, q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]         # Known
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = True

                else:
                    # Slack (single or distributed): known vmag, vang; predict p, q
                    # DISTSLACK: for distributed slack, vmag is known but angle
                    # reference is only soft-fixed; we still use the known angle
                    # from the PF solution as input and predict P, Q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = network.buses_t.v_ang.loc[t, bus]     # Known (reference)
                    pred_p_i    = pred_p[i]
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = True
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = False

                bus_results.loc[t, (bus, "P pu")]      = pred_p_i.item() if torch.is_tensor(pred_p_i) else pred_p_i
                bus_results.loc[t, (bus, "Q pu")]      = pred_q_i.item() if torch.is_tensor(pred_q_i) else pred_q_i
                bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i.item() if torch.is_tensor(pred_vmag_i) else pred_vmag_i
                bus_results.loc[t, (bus, "Angle deg")] = np.degrees(
                    pred_vang_i.item() if torch.is_tensor(pred_vang_i) else pred_vang_i
                )

            if debug:
                print(f"Predicted angles should be small radians: {pred_vang[:3].numpy()}")
                print(f"Predicted angles in degrees: {np.degrees(pred_vang[:3].numpy())}")
                print(f"Angle range: {pred_vang.min():.6f} to {pred_vang.max():.6f} radians")

            # Calculate line flows from predicted voltages
            vmag_np = np.array([
                bus_results.loc[t, (bus, "V pu")] for bus in buses
            ], dtype=float)
            vang_np = np.radians(np.array([
                bus_results.loc[t, (bus, "Angle deg")] for bus in buses
            ], dtype=float))
            # [STSI 270526]:task 028 — direct Δθ for edge_delta models
            if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:
                _fwd_mask = np.zeros(data.edge_index.shape[1], dtype=bool)
                _fwd_mask[::2] = True
                _ei_fwd = data.edge_index[:, _fwd_mask].cpu().numpy()
                _ea_fwd = data.edge_attr[_fwd_mask].cpu().numpy()
                _dt_np = delta_theta_pred.cpu().numpy()
                pred_line_flows = calculate_line_flows_from_delta_theta(
                    delta_theta=_dt_np,
                    vmag=vmag_np,
                    edge_index_fwd=_ei_fwd,
                    edge_attr_fwd=_ea_fwd,
                )
            else:
                pred_line_flows = calculate_line_flows(network, vmag_np, vang_np, t_idx)

            for i, line in enumerate(network.lines.index):
                line_results.loc[t, (line, "P0 pu")]   = pred_line_flows["p0"][i]
                line_results.loc[t, (line, "P1 pu")]   = pred_line_flows["p1"][i]
                line_results.loc[t, (line, "Q0 pu")] = pred_line_flows["q0"][i]
                line_results.loc[t, (line, "Q1 pu")] = pred_line_flows["q1"][i]

    dt_dict = {}  # [STSI 010626]:return delta_theta as 4th element
    if _dt_pred_rows:
        dt_dict["dt_pred"] = np.concatenate(_dt_pred_rows)
        if _dt_true_rows:
            dt_dict["dt_true"] = np.concatenate(_dt_true_rows)
    return bus_results, line_results, prediction_masks, dt_dict


# [COPY of GNN_Powerflow_V2.7_Analysis::_compute_delta_theta_metrics]
def _compute_delta_theta_metrics(vang_pred_bus, dataset, net_idx, t_idx, index_map):  # [STSI 010626]:shared helper for delta_theta error computation across all baselines
    """Compute delta_theta error arrays from per-bus vang prediction.
    
    Computes ground-truth delta_theta on-the-fly from data.y[:,1] (true bus angles)
    and forward_edge_mask. Does NOT depend on data.y_delta_theta (which only exists
    when angle_mode="edge_delta"). This allows all baselines (DC, PyPower, FDLF,
    pandapower) to report delta_theta metrics regardless of dataset angle_mode.
    
    vang_pred_bus: numpy array [n_buses] of predicted voltage angles (radians).
    Returns numpy array of absolute errors per forward edge, or None if lookup fails.
    """
    ds_idx = index_map.get((net_idx, t_idx))
    if ds_idx is None:
        return None
    data = dataset[ds_idx]
    # [STSI 020626]:compute truth from bus angles (always available) instead of
    # y_delta_theta (only set when angle_mode="edge_delta")
    fwd_mask = data.forward_edge_mask.numpy()
    ei = data.edge_index[:, fwd_mask].numpy()
    theta_true = data.y[:, 1].numpy()
    dt_true = theta_true[ei[0]] - theta_true[ei[1]]
    dt_pred = vang_pred_bus[ei[0]] - vang_pred_bus[ei[1]]
    return np.abs(dt_pred - dt_true)


# [COPY of GNN_Powerflow_V2.7_Analysis::postprocess_dangling_buses]
def postprocess_dangling_buses(node_pred, data, p_thresh=1e-4):
    """
    Fix predictions for dangling buses (degree-1 PQ buses with P=Q=0).
    Physics: no current flows → Vmag/Vang must equal neighbor's values.
    Modifies vmag (col 0) and vang (col 1) only; leaves P, Q untouched.
    """
    pred = node_pred.clone()
    x = data.x          # [N,7]: [is_slack, is_pv, is_pq, P, Q, Vmag, Vang]
    ei = data.edge_index  # [2, E]
    N = x.shape[0]

    # Degree (edges appear in both directions in PyG undirected graphs)
    deg = torch.zeros(N, dtype=torch.long, device=x.device) # check for degree-1 buses
    deg.scatter_add_(0, ei[0], torch.ones(ei.shape[1], dtype=torch.long, device=x.device)) # count outgoing edges

    dangling = (deg == 1) & (x[:, 2] > 0.5) & (x[:, 3].abs() < p_thresh) & (x[:, 4].abs() < p_thresh)   # degree-1 PQ buses with P≈0 and Q≈0
    if not dangling.any(): # early exit if no dangling buses
        return pred 

    for i in dangling.nonzero(as_tuple=True)[0].tolist(): # for each dangling bus
        nbrs = ei[1][ei[0] == i]   # check which bus it's connected to (should be exactly one neighbor in an undirected graph)
        nbrs = torch.cat([nbrs, ei[0][ei[1] == i]])  # check both directions to be safe
        nbrs = nbrs[nbrs != i] # exclude self-loop if present
        if len(nbrs) == 0: # 
            continue
        j = nbrs[0].item()  # neighbor index (should be only one)
        pred[i, 0] = pred[j, 0]  # vmag ← neighbour # STSI 26.05.0: this is a simple heuristic fix for dangling buses — we copy the neighbor's Vmag and Vang predictions, since a dangling bus with no load should have the same voltage as its single neighbor. This is a post-processing step that modifies the model's raw predictions to enforce a known physical constraint, and should improve accuracy for these edge cases.
        pred[i, 1] = pred[j, 1]  # vang ← neighbour
    return pred


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_gnn_on_test_set]
def evaluate_gnn_on_test_set(
    model,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnom_share: bool = None,  # None = auto-detect from model.node_embedding.in_features
    device=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None,
    max_buses: int = 0,
    ptdf_offset: int = 0,
    # [STSI260502]: batch_size for batched timing measurement
    batch_size: int = 64,
    # [STSI260502]: flag to enable dangling bus postprocessing
    postprocess_dangling: bool = True,
    verbose: bool = False,
) -> dict:
    """
    Evaluate GNN on test networks snapshot-by-snapshot.

    KEY FIX (2026-03-25): After inference, override known variables with ground
    truth inputs — otherwise scatter plots show model predictions for variables
    the model was never asked to predict (Vmag/Vang at slack/PV), making results
    appear much worse than they are.

    NOTE: vang_mae/vang_rmse returned here are in RADIANS. evaluate_models_on_networks
    reports 'vang_mae_deg' in DEGREES -- do not compare the two directly
    (~57x = 180/pi unit difference).  # [STSI 140926]:clarify unit convention

    Override rules (mirror of _masked_mse_loss):
        PV    → vmag_eval[pv]    = x[pv,    5]  (known Vmag)
        Slack → vmag_eval[slack] = x[slack,  5]  (known Vmag = v_set)
        Slack → vang_eval[slack] = x[slack,  6]  (known Vang = 0)

    STSI 26.04.07: When ptdf_params is not None (learned_edge mode), augments
    data.edge_attr with learned PTDF rows before model forward pass so the
    model receives the same edge feature width it was trained with.

    [STSI260502]: Added multi-level timing:
      solve_time_ms:       pure forward pass per snapshot
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
      batch_solve_time_ms: batched forward pass amortized per snapshot
    Dataset construction and data.to(device) excluded from all timers.

    Returns dict with MAE/RMSE for Vmag, Vang, P, Q + line flow MAE + timing.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    angle_mode = getattr(model, "angle_mode", "node")  # [STSI 260526]:auto-detect angle_mode
    theta_recon_times = []  # [STSI 260526]:timing accumulator for edge_delta
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
        angle_mode=angle_mode,  # [STSI 260526]:pass angle_mode
        ptdf_branch_mode="all" if angle_mode in ("edge_delta", "both") else "lines",  # [STSI 260526]:edge_delta needs all branches for BFS connectivity
    )

    all_vmag_err, all_vang_err, all_p_err, all_q_err = [], [], [], []  # [STSI 010626]:element-wise error lists for true RMSE
    vmag_true_all, vang_true_all, p_true_all, q_true_all = [], [], [], []

    all_lflow_err = []  # [STSI 010626]:element-wise line flow errors
    all_lflow_q0_err = []
    all_true_flows, all_pred_flows = [], []
    all_true_q0_flows, all_pred_q0_flows = [], []
    all_delta_theta_errors = []  # [STSI 270526]:task 028 — collect per-edge Δθ errors
    line_flow_errors_node_theta = []  # [STSI 030626]:dual-head comparison — node-θ line flow errors
    theta_consistency_errors = []  # [STSI 030626]:dual-head comparison — node-θ vs BFS-reconstructed-θ
    n_snapshots = 0

    # [STSI260502]: per-snapshot timing lists (solve and postproc measured separately)
    solve_times = []
    postproc_times = []
    # [STSI260502]: collect prepared data for batched timing later
    all_data_for_batch = []

    with torch.no_grad():
        for i in range(len(dataset)):
            data = dataset[i].to(device)

            # STSI 26.04.07: augment edge_attr for learned_edge models
            net_idx, _ = dataset._index[i]
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset, device)

            # [STSI260502]: save a copy for batched timing (before forward modifies anything)
            all_data_for_batch.append(data.clone())

            # [STSI260502]: Level 1 — time pure forward pass only
            t_solve_start = time.perf_counter()
            out = model(data, return_embeddings=True)
            node_pred = out[0]
            delta_theta_pred = out[3] if len(out) > 3 else None  # [STSI 260526]:4th elem in embeddings mode
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t_solve_start
            # [STSI 240726]: start postproc timer BEFORE θ reconstruction so
            # total_time_ms includes θ recon (was leaking into theta_recon_times only)
            t_post_start = time.perf_counter()

            # θ reconstruction for edge_delta mode [STSI 260526]
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                t_recon_start = time.perf_counter()
                theta_recon = reconstruct_theta_from_path_matrix(  # [STSI 240726]:replaced BFS loop with matmul via path-incidence matrix
                    delta_theta_pred,
                    data.bfs_path_matrix,
                )
                theta_recon_times.append((time.perf_counter() - t_recon_start) * 1000)
                # Assemble [N, 4] from [N, 3] + reconstructed θ
                node_pred_full = torch.zeros(data.x.size(0), 4, device=device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = theta_recon       # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P
                node_pred_full[:, 3] = node_pred[:, 2]  # Q
                node_pred = node_pred_full

            # ── Extract masks ─────────────────────────────────────────────────
            slack_mask = data.slack_mask.bool()
            pv_mask    = data.pv_mask.bool()
            pq_mask    = data.pq_mask.bool()
            
            # ── Dangling bus post-processing ──────────────────────────────────
            if postprocess_dangling:
                node_pred = postprocess_dangling_buses(node_pred, data)
            
            # ── Raw predictions ───────────────────────────────────────────────
            vmag_eval = node_pred[:, 0].clone()
            vang_eval = node_pred[:, 1].clone()
            p_eval    = node_pred[:, 2].clone()
            q_eval    = node_pred[:, 3].clone()

            # ── Override known variables with ground-truth inputs ─────────────
            # PV: Vmag is known (set by voltage regulator)
            if pv_mask.any():
                vmag_eval[pv_mask] = data.x[pv_mask, 5]

            # Slack: both Vmag and Vang=0 are known
            if slack_mask.any():
                vmag_eval[slack_mask] = data.x[slack_mask, 5]
                vang_eval[slack_mask] = data.x[slack_mask, 6]  # should be ~0

            # PQ: P and Q are known inputs
            if pq_mask.any():
                p_eval[pq_mask] = data.x[pq_mask, 3]
                q_eval[pq_mask] = data.x[pq_mask, 4]

            # PV: P is known input
            if pv_mask.any():
                p_eval[pv_mask] = data.x[pv_mask, 3]

            # ── Ground truth ──────────────────────────────────────────────────
            vmag_true = data.y[:, 0]
            vang_true = data.y[:, 1]
            p_true    = data.y[:, 2]
            q_true    = data.y[:, 3]

            all_vmag_err.extend((vmag_eval - vmag_true).abs().cpu().numpy())  # [STSI 010626]:element-wise errors for correct RMSE
            all_vang_err.extend((vang_eval - vang_true).abs().cpu().numpy())
            all_p_err.extend((p_eval - p_true).abs().cpu().numpy())
            all_q_err.extend((q_eval - q_true).abs().cpu().numpy())
            vmag_true_all.append(vmag_true.detach())
            vang_true_all.append(vang_true.detach())
            p_true_all.append(p_true.detach())
            q_true_all.append(q_true.detach())
            n_snapshots += 1

            # ── Step 4 (partial): line flow MAE ──────────────────────────────
            if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                try:
                    net_idx, t_idx = dataset._index[i]
                    network  = test_networks[net_idx]
                    snapshot = network.snapshots[t_idx]
                    if not network.lines_t.p0.empty:
                        true_flows = network.lines_t.p0.loc[snapshot].values
                        vmag_np = vmag_eval.cpu().numpy()
                        vang_np = vang_eval.cpu().numpy()
                        # [STSI 270526]:task 028 — direct Δθ line flows for edge_delta models
                        if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:
                            _data_cpu = data.cpu()
                            _all_fwd_mask_np = np.zeros(_data_cpu.edge_index.shape[1], dtype=bool)
                            _all_fwd_mask_np[::2] = True
                            _ei_fwd_np = _data_cpu.edge_index[:, _all_fwd_mask_np].numpy()
                            _ea_fwd_np = _data_cpu.edge_attr[_all_fwd_mask_np].numpy()
                            _dt_pred_np = delta_theta_pred.cpu().numpy()
                            flows = calculate_line_flows_from_delta_theta(
                                delta_theta=_dt_pred_np,
                                vmag=vmag_np,
                                edge_index_fwd=_ei_fwd_np,
                                edge_attr_fwd=_ea_fwd_np,
                            )
                        else:
                            # [STSI 240726]:replaced pandas .loc loop with vectorized Δθ calc (avoids ~6ms/snap overhead)
                            _fwd_mask_np = np.zeros(data.edge_index.shape[1], dtype=bool)
                            _fwd_mask_np[::2] = True
                            _ei_fwd_np = data.edge_index[:, _fwd_mask_np].cpu().numpy()
                            _ea_fwd_np = data.edge_attr[_fwd_mask_np].cpu().numpy()
                            _dt_np = vang_np[_ei_fwd_np[0]] - vang_np[_ei_fwd_np[1]]
                            flows = calculate_line_flows_from_delta_theta(
                                delta_theta=_dt_np,
                                vmag=vmag_np,
                                edge_index_fwd=_ei_fwd_np,
                                edge_attr_fwd=_ea_fwd_np,
                            )
                        n_lines = len(network.lines)
                        pred_flows = np.array(flows["p0"])[:n_lines]  # lines only (excl transformers)
                        all_lflow_err.extend(np.abs(pred_flows - true_flows))  # [STSI 010626]:element-wise
                        all_true_flows.append(true_flows)
                        all_pred_flows.append(pred_flows)
                    if not network.lines_t.q0.empty:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])[:n_lines]  # lines only
                        all_lflow_q0_err.extend(np.abs(pred_q0 - true_q0))  # [STSI 010626]:element-wise
                        all_true_q0_flows.append(true_q0)
                        all_pred_q0_flows.append(pred_q0)
                except Exception as _lfe:
                    if not all_lflow_err:  # only warn once
                        print(f"[line_flow_mae] skipped: {_lfe}")

            # [STSI 270526]:task 028 — collect per-edge Δθ errors for edge_delta models
            if angle_mode in ("edge_delta", "both") and hasattr(data, "y_delta_theta"):
                _dt_true = data.y_delta_theta.cpu().numpy()
                _dt_pred = (delta_theta_pred.cpu().numpy()
                            if delta_theta_pred is not None
                            else np.zeros_like(_dt_true))
                all_delta_theta_errors.append(np.abs(_dt_pred - _dt_true))

            # ── Dual-head θ comparison (angle_mode=="both" only) ──────────── [STSI 030626]
            if angle_mode == "both" and delta_theta_pred is not None:
                # 1. Line flows from NODE head θ (using θ_i − θ_j internally)
                if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                    try:
                        _net_idx_dh, _t_idx_dh = dataset._index[i]
                        _network_dh = test_networks[_net_idx_dh]
                        _snapshot_dh = _network_dh.snapshots[_t_idx_dh]
                        if not _network_dh.lines_t.p0.empty:
                            _true_flows_dh = _network_dh.lines_t.p0.loc[_snapshot_dh].values
                            _vmag_np_dh = vmag_eval.cpu().numpy()
                            _vang_np_dh = vang_eval.cpu().numpy()  # node-head θ
                            _flows_node = calculate_line_flows(_network_dh, _vmag_np_dh, _vang_np_dh)
                            _n_lines_dh = len(_network_dh.lines)
                            _pred_flows_node = np.array(_flows_node["p0"])[:_n_lines_dh]
                            line_flow_errors_node_theta.extend(np.abs(_pred_flows_node - _true_flows_dh))
                    except Exception:
                        pass

                # 2. θ consistency: node head θ vs BFS-reconstructed θ from edge Δθ
                _theta_recon_dh = reconstruct_theta_from_path_matrix(  # [STSI 240726]:use vectorized matmul recon
                    delta_theta_pred,
                    data.bfs_path_matrix,
                )
                _node_theta = vang_eval.cpu().numpy()
                _recon_theta = _theta_recon_dh.cpu().numpy()
                _slack_idx_dh = int(data.slack_mask.bool().nonzero(as_tuple=True)[0][0])
                _non_slack = np.ones(len(_node_theta), dtype=bool)
                _non_slack[_slack_idx_dh] = False
                theta_consistency_errors.extend(np.abs(_node_theta[_non_slack] - _recon_theta[_non_slack]))

            # [STSI260502]: end post-processing timer (includes masks + line flows)
            t_post = time.perf_counter() - t_post_start
            solve_times.append(t_solve)
            postproc_times.append(t_post)

    # [STSI260502]: Phase 2 — batched timing (forward + PARALLEL postproc)
    # [STSI 240726]: postproc now timed on the batched output (one unit per chunk),
    # amortized per snapshot. batch_solve_time_ms = forward + parallel postproc.
    n_batch_warmup = 3
    _EXCLUDE_BATCH = ['y_ptdf', 'ptdf_line_index', 'y_line_p',
                      'bfs_edge_idx', 'bfs_signs','bfs_path_matrix', 'bfs_node_order',
                      'bfs_parent', 'y_delta_theta']  # variable-size keys excluded from PyG batch
    with torch.no_grad():
        if all_data_for_batch:
            # Warmup batched forward (optionally run one postproc pass too, to
            # warm numpy / .cpu() paths so the first timed chunk isn't cold)
            for _ in range(n_batch_warmup):
                _wb = Batch.from_data_list(all_data_for_batch[:batch_size], exclude_keys=_EXCLUDE_BATCH)
                _ = model(_wb)

            batch_fwd_times, batch_pp_times, batch_counts = [], [], []
            for start in range(0, len(all_data_for_batch), batch_size):
                chunk = all_data_for_batch[start:start + batch_size]
                bp_batch = Batch.from_data_list(chunk, exclude_keys=_EXCLUDE_BATCH)

                # ── Level 1: batched forward (timed) ──────────────
                t_fwd = time.perf_counter()
                bp_out = model(bp_batch)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_fwd_times.append(time.perf_counter() - t_fwd)

                bp_node = bp_out[0] if isinstance(bp_out, tuple) else bp_out
                bp_dt   = bp_out[2] if isinstance(bp_out, tuple) and len(bp_out) > 2 else None

                # ── Level 2: batched PARALLEL postproc (timed) ────────
                t_pp = time.perf_counter()

                # (a) θ reconstruction — edge_delta only, per-graph matmul via path-incidence matrix  # [STSI 240726]:replaced BFS loop with per-graph matmul
                if angle_mode == "edge_delta" and bp_dt is not None:
                    vang_stacked = torch.zeros(bp_node.size(0), device=device)
                    node_off, dt_off = 0, 0
                    for g_data in chunk:
                        n_g = g_data.x.size(0)
                        e_fwd_g = g_data.edge_index.size(1) // 2  # [::2] forward edges
                        theta_g = reconstruct_theta_from_path_matrix(
                            bp_dt[dt_off:dt_off + e_fwd_g],
                            g_data.bfs_path_matrix,  # [n_g, e_fwd_g] — one matmul, no scalar loop
                        )
                        vang_stacked[node_off:node_off + n_g] = theta_g
                        node_off += n_g
                        dt_off   += e_fwd_g
                    # bp_node is [N,3]=(Vmag,P,Q) for edge_delta → assemble [N,4]
                    bp_eval = torch.zeros(bp_node.size(0), 4, device=device)
                    bp_eval[:, 0] = bp_node[:, 0]
                    bp_eval[:, 1] = vang_stacked
                    bp_eval[:, 2] = bp_node[:, 1]
                    bp_eval[:, 3] = bp_node[:, 2]
                else:
                    bp_eval = bp_node  # [N,4] for node / both

                # (b) dangling-bus fix (works on batched graph — no cross-graph edges)
                if postprocess_dangling:
                    bp_eval = postprocess_dangling_buses(bp_eval, bp_batch)

                # (c) vectorized mask overrides on the stacked node tensor
                vmag_b = bp_eval[:, 0].clone()
                vang_b = bp_eval[:, 1].clone()
                _sm = bp_batch.slack_mask.bool()
                _pv = bp_batch.pv_mask.bool()
                if _pv.any():
                    vmag_b[_pv] = bp_batch.x[_pv, 5]
                if _sm.any():
                    vmag_b[_sm] = bp_batch.x[_sm, 5]
                    vang_b[_sm] = bp_batch.x[_sm, 6]

                # (d) PARALLEL line flows: one vectorized call over the whole chunk
                #     (batched edge_index already carries GLOBAL node offsets)
                _fwd = torch.zeros(bp_batch.edge_index.size(1), dtype=torch.bool, device=device)
                _fwd[::2] = True
                ei_fwd  = bp_batch.edge_index[:, _fwd].cpu().numpy()
                ea_fwd  = bp_batch.edge_attr[_fwd].cpu().numpy()
                vmag_np = vmag_b.cpu().numpy()
                if angle_mode in ("edge_delta", "both") and bp_dt is not None:
                    dt_np = bp_dt.cpu().numpy()             # aligned with edge_index[:, ::2]
                else:
                    vang_np = vang_b.cpu().numpy()
                    dt_np = vang_np[ei_fwd[0]] - vang_np[ei_fwd[1]]
                _ = calculate_line_flows_from_delta_theta(
                    delta_theta=dt_np, vmag=vmag_np,
                    edge_index_fwd=ei_fwd, edge_attr_fwd=ea_fwd,
                )

                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_pp_times.append(time.perf_counter() - t_pp)
                batch_counts.append(len(chunk))

            _cnt = max(1, sum(batch_counts))
            batch_inference_only_ms = sum(batch_fwd_times) * 1000 / _cnt
            batch_postproc_ms       = sum(batch_pp_times)  * 1000 / _cnt
        else:
            batch_inference_only_ms = float('nan')
            batch_postproc_ms       = float('nan')

    metrics = {  # [STSI 010626]:element-wise MAE and RMSE (flat list pattern)
        "vmag_mae":      float(np.mean(all_vmag_err)),
        "vang_mae":      float(np.mean(all_vang_err)),
        "p_mae":         float(np.mean(all_p_err)),
        "q_mae":         float(np.mean(all_q_err)),
        "vmag_rmse":     float(np.sqrt(np.mean(np.array(all_vmag_err)**2))),
        "vang_rmse":     float(np.sqrt(np.mean(np.array(all_vang_err)**2))),
        "vmag_true_std": float(torch.cat(vmag_true_all).std().item()),
        "vang_true_std": float(torch.cat(vang_true_all).std().item()),
        "p_true_std":    float(torch.cat(p_true_all).std().item()),
        "q_true_std":    float(torch.cat(q_true_all).std().item()),
    }
    if all_lflow_err:  # [STSI 010626]:element-wise RMSE
        metrics["line_flow_mae"] = float(np.mean(all_lflow_err))
        metrics["line_flow_rmse"] = float(np.sqrt(np.mean(np.array(all_lflow_err)**2)))
    if all_lflow_q0_err:
        metrics["line_flow_q0_mae"] = float(np.mean(all_lflow_q0_err))
        metrics["line_flow_q0_rmse"] = float(np.sqrt(np.mean(np.array(all_lflow_q0_err)**2)))
    metrics["p_rmse"] = float(np.sqrt(np.mean(np.array(all_p_err)**2)))
    metrics["q_rmse"] = float(np.sqrt(np.mean(np.array(all_q_err)**2)))

    if all_true_flows:
        raw = {
            "p0_true": np.concatenate(all_true_flows),
            "p0_pred": np.concatenate(all_pred_flows),
        }
        if all_true_q0_flows:
            raw["q0_true"] = np.concatenate(all_true_q0_flows)
            raw["q0_pred"] = np.concatenate(all_pred_q0_flows)
        metrics["line_flow_raw"] = raw
    if verbose:     
        print("----------------\n Evaluation Metrics (known variables overridden) ---------------")
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f"  {k:20s}: {v:.6f}")

    # [STSI260502]: monolithic timing kept for backward compat
    inference_time_s = sum(solve_times) + sum(postproc_times)
    metrics["inference_time_s"] = inference_time_s
    metrics["inference_time_per_snapshot_ms"] = (
        1000.0 * inference_time_s / max(1, n_snapshots)
    )
    
    # [STSI260502]: new multi-level timing metrics
    solve_ms    = float(np.mean(solve_times)) * 1000
    postproc_ms = float(np.mean(postproc_times)) * 1000
    metrics["solve_time_ms"]       = solve_ms
    metrics["postproc_time_ms"]    = postproc_ms
    metrics["total_time_ms"]       = solve_ms + postproc_ms
    # [STSI 240726]: batch_solve_time_ms is now the PARALLEL total (batched forward
    # + batched postproc), overwriting the 067 conservative (forward + serial) total.
    metrics["batch_inference_only_time_ms"] = batch_inference_only_ms
    metrics["batch_postproc_time_ms"]       = batch_postproc_ms  # diagnostic; NOT in REQUIRED_METRICS
    metrics["batch_solve_time_ms"]          = batch_inference_only_ms + batch_postproc_ms
    if theta_recon_times:  # [STSI 260526]:edge_delta timing metric
        metrics["theta_reconstruction_time_ms"] = float(np.mean(theta_recon_times))
    if all_delta_theta_errors:  # [STSI 270526]:task 028 — Δθ accuracy metrics
        _all_dt_err = np.concatenate(all_delta_theta_errors)
        metrics["delta_theta_mae"] = float(np.mean(_all_dt_err))
        metrics["delta_theta_rmse"] = float(np.sqrt(np.mean(_all_dt_err**2)))
    # [STSI 030626]:dual-head θ consistency metrics (angle_mode=="both" only)
    if angle_mode == "both":
        if all_lflow_err:
            metrics["line_flow_rmse_delta_theta"] = metrics.get("line_flow_rmse", float("nan"))
            metrics["line_flow_mae_delta_theta"] = metrics.get("line_flow_mae", float("nan"))
        if line_flow_errors_node_theta:
            _arr_nt = np.array(line_flow_errors_node_theta)
            metrics["line_flow_rmse_node_theta"] = float(np.sqrt(np.mean(_arr_nt**2)))
            metrics["line_flow_mae_node_theta"] = float(np.mean(_arr_nt))
        if theta_consistency_errors:
            _arr_tc = np.array(theta_consistency_errors)
            metrics["theta_consistency_mae"] = float(np.mean(_arr_tc))
            metrics["theta_consistency_rmse"] = float(np.sqrt(np.mean(_arr_tc**2)))
    if verbose:
        print(f"\n  --- Timing breakdown (per snapshot) ---")
        print(f"  {'solve_time_ms':20s}: {solve_ms:.3f}")
        print(f"  {'postproc_time_ms':20s}: {postproc_ms:.3f}")
        print(f"  {'total_time_ms':20s}: {solve_ms + postproc_ms:.3f}")
        print(f"  {'batch_fwd_only_ms':20s}: {batch_inference_only_ms:.3f}")
        print(f"  {'batch_postproc_ms':20s}: {batch_postproc_ms:.3f}")
        print(f"  {'batch_solve_time_ms':20s}: {(batch_inference_only_ms + batch_postproc_ms):.3f}")

    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::load_sweep_results]
def load_sweep_results(pkl_path: str) -> list:
    """
    Load a full sweep result list from a pickle file.
    Tags each run with _source_pkl for save_backfilled traceability.
    """
    with open(pkl_path, "rb") as f:
        runs = pickle.load(f)
    for r in runs:
        r["_source_pkl"] = pkl_path
    print(f"Loaded {len(runs)} runs from {pkl_path}")
    return runs


# [STSI 141026]:verbatim copies extracted programmatically (task 088); do not edit, originals are ground truth

In [ ]:
from pypower.api import runpf, ppoption
from pypower.idx_bus import (BUS_I, BUS_TYPE, PD, QD, GS, BS,
                              BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN)
from pypower.idx_gen import (GEN_BUS, PG, QG, QMAX, QMIN, VG,
                              MBASE, GEN_STATUS, PMAX, PMIN)
from pypower.idx_brch import (F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A,
                               RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, PF, QF, PT, QT)


# [COPY of GNN_Powerflow_V2.7_Analysis::pypsa_to_ppc]
def pypsa_to_ppc(network, snapshot, base_mva=1.0):
    """Convert a PyPSA network at a single snapshot to a PyPower ppc dict.
    
    Assumes network values are already in p.u. (v_nom=1.0, line lengths=0).
    All outputs remain in p.u. with the given base_mva.
    
    Returns:
        ppc, bus_names, bus_idx, line_names, trafo_names
    """
    bus_names = network.buses.index.tolist()
    bus_idx = {name: i + 1 for i, name in enumerate(bus_names)}  # 1-indexed
    n = len(bus_names)

    # Bus type sets from generators
    slack_buses = set(network.generators[network.generators['control'] == 'Slack']['bus'])
    pv_buses = set(network.generators[network.generators['control'] == 'PV']['bus'])

    # Aggregate loads per bus for this snapshot
    lp = network.loads_t.p_set.loc[snapshot]
    lq = network.loads_t.q_set.loc[snapshot]
    bus_p, bus_q = {}, {}
    for lname, ldata in network.loads.iterrows():
        b = ldata['bus']
        bus_p[b] = bus_p.get(b, 0.0) + lp.get(lname, 0.0)
        bus_q[b] = bus_q.get(b, 0.0) + lq.get(lname, 0.0)

    # --- Bus matrix (13 cols) ---
    bus_mat = np.zeros((n, 13))
    for i, bname in enumerate(bus_names):
        btype = 3 if bname in slack_buses else (2 if bname in pv_buses else 1)
        bus_mat[i, BUS_I]    = i + 1
        bus_mat[i, BUS_TYPE] = btype
        bus_mat[i, PD]       = bus_p.get(bname, 0.0)
        bus_mat[i, QD]       = bus_q.get(bname, 0.0)
        bus_mat[i, VM]       = 1.0
        bus_mat[i, VA]       = 0.0
        bus_mat[i, BASE_KV]  = network.buses.loc[bname, 'v_nom']
        bus_mat[i, VMAX]     = 1.1
        bus_mat[i, VMIN]     = 0.9

    # Get solved generator outputs for this snapshot
    gen_p_solved, gen_q_solved = {}, {}
    if len(network.generators_t.p) > 0:
        row_gp = network.generators_t.p.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gp.index:
                gen_p_solved[gname] = float(row_gp[gname])
    if hasattr(network.generators_t, 'q') and len(network.generators_t.q) > 0:
        row_gq = network.generators_t.q.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gq.index:
                gen_q_solved[gname] = float(row_gq[gname])

    # --- Gen matrix (10 cols) ---
    gen_rows = []
    for gname, gdata in network.generators.iterrows():
        b = gdata['bus']
        vg = float(network.buses_t.v_mag_pu.loc[snapshot, b])
        row = np.zeros(10)
        row[GEN_BUS]    = bus_idx[b]
        row[VG]         = vg
        row[MBASE]      = base_mva
        row[GEN_STATUS] = 1
        row[QMAX] = 9999.0;  row[QMIN] = -9999.0
        row[PMAX] = 9999.0;  row[PMIN] = -9999.0
        if gdata['control'] != 'Slack':
            row[PG] = gen_p_solved.get(gname, 0.0)  # PV: fixed P, pypower solves Q
        if gdata['control'] == 'PQ':
            row[QG] = gen_q_solved.get(gname, 0.0)  # PQ: both fixed
        gen_rows.append(row)

    gen_mat = np.array(gen_rows)

    # --- Branch matrix — lines ---
    branch_rows = []
    line_names = network.lines.index.tolist()
    for lname, line in network.lines.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[line['bus0']]
        row[T_BUS]     = bus_idx[line['bus1']]
        row[BR_R]      = line['r']
        row[BR_X]      = line['x']
        row[BR_B]      = line['b']
        row[RATE_A]    = line['s_nom']
        row[BR_STATUS] = 1
        row[TAP]       = 0.0   # 0 → treated as 1.0 by pypower
        branch_rows.append(row)

    # --- Branch matrix — transformers (use raw r/x, not r_pu/x_pu) ---
    trafo_names = network.transformers.index.tolist()
    for tname, tr in network.transformers.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[tr['bus0']]
        row[T_BUS]     = bus_idx[tr['bus1']]
        row[BR_R]      = tr['r']
        row[BR_X]      = tr['x']
        row[BR_B]      = tr.get('b', 0.0)
        row[RATE_A]    = tr['s_nom']
        row[TAP]       = tr['tap_ratio'] if tr['tap_ratio'] != 0 else 1.0
        row[SHIFT]     = tr['phase_shift']
        row[BR_STATUS] = 1
        branch_rows.append(row)

    branch_mat = np.array(branch_rows)

    ppc = {'version': '2', 'baseMVA': base_mva,
           'bus': bus_mat, 'gen': gen_mat, 'branch': branch_mat}

    return ppc, bus_names, bus_idx, line_names, trafo_names


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_pypower_baseline]
def evaluate_pypower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run PyPower AC PF on all networks/snapshots, compare to PyPSA ground truth.
    
    All comparisons are in p.u.  Returns dict compatible with scatter plot functions:
      vmag_mae, vang_mae, p_mae, q_mae, line_flow_mae, line_flow_q0_mae,
      inference_time_per_snapshot_ms, n_snapshots, n_converged, n_failed
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        # Check which time-series exist in ground truth
        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            # --- Convert & ----
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)
            # --- Solve (timed) ---
            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            # Sort result buses by index
            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            # --- Voltage comparison (p.u. / radians) ---
            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            # --- Bus P/Q injection (p.u.) ---
            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            # --- Line flows (p.u.) — lines only, first n_lines branches ---
            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"PyPower AC baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_fdlf_baseline]
def evaluate_fdlf_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run PyPower Fast Decoupled (XB) PF on all networks/snapshots.
    Identical to evaluate_pypower_baseline but uses PF_ALG=2 (FDLF-XB).
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=2)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)

            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"FDLF (XB) baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_pandapower_baseline]
def evaluate_pandapower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run pandapower AC NR PF on all networks/snapshots.
    Path: pypsa_to_ppc → from_ppc (untimed) → runpp (timed).
    With baseMVA=1.0, pandapower MW/MVAr values equal p.u. numerically.
    """
    import pandapower as pdp
    from pandapower.converter import from_ppc
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            # Conversion (untimed — same overhead as any solver needing its own format)
            ppc, _, _, _, _ = pypsa_to_ppc(network, snap, base_mva)
            try:
                pp_net = from_ppc(ppc, f_hz=50.0)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] from_ppc failed: {e}")
                continue

            # Solve (timed)
            t0 = time.perf_counter()
            try:
                pdp.runpp(pp_net, algorithm='nr', init='flat', numba=True)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] runpp failed: {e}")
                continue
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not pp_net.converged:
                n_failed += 1
                continue
            n_converged += 1

            # Results — baseMVA=1 so MW == p.u. numerically
            pp_vmag = pp_net.res_bus.vm_pu.values[:n_buses]
            pp_vang = np.deg2rad(pp_net.res_bus.va_degree.values[:n_buses])

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            # Bus P/Q: res_bus.p_mw is net injection (gen-load) in MW (= p.u. @ base=1)
            if has_bus_p:
                pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                pp_p = -pp_net.res_bus.p_mw.values[:n_buses]
                all_p_err.extend(np.abs(pypsa_p - pp_p))
            if has_bus_q:
                pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                pp_q = -pp_net.res_bus.q_mvar.values[:n_buses]
                all_q_err.extend(np.abs(pypsa_q - pp_q))

            # Line flows
            n_res_lines = len(pp_net.res_line)
            if has_line_p and n_res_lines >= n_lines:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                pp_lp = pp_net.res_line.p_from_mw.values[:n_lines]
                all_line_p_err.extend(np.abs(pypsa_lp - pp_lp))
            if has_line_q and n_res_lines >= n_lines:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                pp_lq = pp_net.res_line.q_from_mvar.values[:n_lines]
                all_line_q_err.extend(np.abs(pypsa_lq - pp_lq))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"pandapower NR baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::build_ref_baselines]
def build_ref_baselines(networks, n_sample=None, ref_single=None, ref_batch=None, test_networks=None):  # [STSI 140926]:n_sample=None -> full test split; test_networks override
    """
    Build (or incrementally fill) ref_single and ref_batch baseline timing dicts.

    Parameters
    ----------
    networks  : full list of networks (train+val+test); test split reconstructed internally.
                Ignored if test_networks is provided.
    n_sample  : number of test networks to time. Default None = use the full test
                population (same population evaluate_gnn_on_test_set is run on for
                the GNN runs these baselines will be plotted against). Pass a smaller
                int to trade accuracy of the comparison for faster iteration — a
                warning is printed whenever the resolved sample is smaller than the
                full test population.
    ref_single : existing dict to fill; keys already present are skipped
    ref_batch  : existing dict to fill; keys already present are skipped
    test_networks : optional pre-selected test-network list to use directly, bypassing
                internal reconstruct_test_split(networks). Use this when `networks` is
                already a curated test set (e.g. a single-topology exploration list) —
                passing such a list as `networks` would otherwise be re-split 70/15/15,
                collapsing the effective sample size unexpectedly.

    Returns
    -------
    (ref_single, ref_batch)
    """
    if ref_single is None:
        ref_single = {}
    if ref_batch is None:
        ref_batch = {}

    if test_networks is not None:
        test_nets = list(test_networks)
    else:
        _, _, test_nets = reconstruct_test_split(networks)

    sample = test_nets if n_sample is None else test_nets[:n_sample]
    if len(sample) < len(test_nets):
        print(
            f"WARNING: baseline timing uses {len(sample)}/{len(test_nets)} test networks "
            f"— GNN runs plotted alongside these baselines are typically evaluated on the "
            f"full {len(test_nets)}-network test set. Metrics are not directly comparable "
            f"unless n_sample matches (or is None)."
        )
    print(f"Using {len(sample)} test networks for baseline timing.")

    # single-snapshot baselines
    if "PyPower AC" not in ref_single:
        print("Running PyPower AC...")
        ref_single["PyPower AC"] = evaluate_pypower_baseline(sample, test_networks=sample)  # [STSI 010626]:pass test_networks for delta_theta
    if "PyPSA single" not in ref_single:
        print("Running PyPSA single...")
        ref_single["PyPSA single"] = evaluate_pypsa_timing_single(sample)
    if "FDLF" not in ref_single:
        print("Running FDLF...")
        ref_single["FDLF"] = evaluate_fdlf_baseline(sample, test_networks=sample)
    if "pandapower" not in ref_single:
        print("Running pandapower...")
        ref_single["pandapower"] = evaluate_pandapower_baseline(sample, test_networks=sample)

    # batch baselines
    if "PyPSA batch" not in ref_batch:
        print("Running PyPSA batch...")
        ref_batch["PyPSA batch"] = evaluate_pypsa_timing(sample)
    if "DC lpf" not in ref_batch:
        print("Running DC lpf...")
        ref_batch["DC lpf"] = evaluate_dc_baseline(sample)

    print("\nref_single timings (ms/snap):")
    for k, v in ref_single.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")
    print("ref_batch timings (ms/snap):")
    for k, v in ref_batch.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")

    return ref_single, ref_batch


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_pypsa_timing]
def evaluate_pypsa_timing(networks, verbose=True):
    """Measure PyPSA AC PF wall time by re-running pf() on a network copy.
    
    Runs pf() on all snapshots at once per network (same as normal usage),
    divides by snapshot count for per-snapshot average.
    Error = 0 by definition (PyPSA is the ground truth).
    """
    all_times_ms = []
    for network in networks:
        net_copy = network.copy()
        t0 = time.perf_counter()
        net_copy.pf(use_seed=True)
        t1 = time.perf_counter()
        ms_per_snap = (t1 - t0) * 1000.0 / max(1, len(network.snapshots))
        all_times_ms.append(ms_per_snap)

    avg_ms = float(np.mean(all_times_ms))
    metrics = {  # [STSI 010626]:add all zero-valued keys so PyPSA appears on all scatter subplots
        'vmag_mae': 0.0, 'vmag_rmse': 0.0,
        'vang_mae': 0.0, 'vang_rmse': 0.0,
        'p_mae': 0.0, 'p_rmse': 0.0,
        'q_mae': 0.0, 'q_rmse': 0.0,
        'line_flow_mae': 0.0, 'line_flow_rmse': 0.0,
        'line_flow_q0_mae': 0.0, 'line_flow_q0_rmse': 0.0,
        'delta_theta_mae': 0.0, 'delta_theta_rmse': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
        'n_snapshots': sum(len(n.snapshots) for n in networks),
    }
    if verbose:
        print(f"PyPSA AC timing: {avg_ms:.3f} ms/snap avg over {len(networks)} network(s)")
    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_pypsa_timing_single]
def evaluate_pypsa_timing_single(networks, verbose=True):
    """Measure PyPSA AC PF time per individual snapshot (no batching)."""
    times_ms = []
    for net in networks:
        for snap in net.snapshots:
            net_copy = net.copy()
            net_copy.set_snapshots([snap])
            t0 = time.perf_counter()
            net_copy.pf(use_seed=True)
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000)
    
    avg_ms = float(np.mean(times_ms))
    metrics = {  # [STSI 010626]:add all zero-valued keys so PyPSA appears on all scatter subplots
        'vmag_mae': 0.0, 'vmag_rmse': 0.0,
        'vang_mae': 0.0, 'vang_rmse': 0.0,
        'p_mae': 0.0, 'p_rmse': 0.0,
        'q_mae': 0.0, 'q_rmse': 0.0,
        'line_flow_mae': 0.0, 'line_flow_rmse': 0.0,
        'line_flow_q0_mae': 0.0, 'line_flow_q0_rmse': 0.0,
        'delta_theta_mae': 0.0, 'delta_theta_rmse': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
    }
    if verbose:
        print(f"PyPSA AC (single): {avg_ms:.3f} ms/snap over {len(times_ms)} snapshots")
    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::evaluate_dc_baseline]
def evaluate_dc_baseline(
    test_networks,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Run PyPSA linear (DC) power flow on all test-set snapshots and compute
    the same MAE metrics as evaluate_gnn_on_test_set.

    DC capabilities:
        vmag:          flat 1.0 p.u. (DC assumption) — error at PQ buses
        vang:          solved by lpf()               — error at PQ+PV buses
        p injections:  lossless balance              — error at Slack bus
        q injections:  naive zero guess (DC has no Q model) — error at PV+Slack buses
        line_flow_mae: p0 from lpf() vs AC p0        — valid comparison
        line_flow_q0:  naive zero guess (DC has no Q model) — same treatment as q injections

    Both Q-related metrics are a zero-guess, not a real DC capability — plotting
    code (plot_timing_vs_accuracy) marks DC's Q-metric points with a distinct
    marker to avoid implying DC actually solves for reactive power.

    NOTE: vang_mae/vang_rmse returned here are in RADIANS. evaluate_models_on_networks
    reports 'vang_mae_deg' in DEGREES -- do not compare the two directly
    (~57x = 180/pi unit difference).  # [STSI 140926]:clarify unit convention

    Returns a dict compatible with evaluate_gnn_on_test_set output.
    """
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    index_map = {(ni, ti): k for k, (ni, ti) in enumerate(dataset._index)}

    all_vmag_err, all_vang_err, all_p_err, all_q_err = [], [], [], []  # [STSI 010626]:element-wise errors for correct RMSE
    all_lflow_err = []
    all_lflow_q0_err = []  # naive Q=0 guess, mirrors node-level q_dc=0 treatment
    all_dt_errors = []
    n_snapshots = 0
    t_solve = 0.0

    for net_idx, network in enumerate(test_networks):
        if verbose and net_idx % 20 == 0:
            print(f"  [DC] network {net_idx + 1}/{len(test_networks)}")

        net_copy = network.copy()
        t0 = time.time()
        try:
            net_copy.lpf()
        except Exception as e:
            if verbose:
                print(f"  [DC] lpf() failed on network {net_idx}: {e}")
            continue
        t_solve += time.time() - t0

        buses = network.buses.index

        for t_idx, snap in enumerate(network.snapshots):
            ds_idx = index_map.get((net_idx, t_idx))
            if ds_idx is None:
                continue

            data       = dataset[ds_idx]
            slack_mask = data.slack_mask.bool().numpy()
            pv_mask    = data.pv_mask.bool().numpy()
            pq_mask    = data.pq_mask.bool().numpy()
            x_np       = data.x.numpy()
            y_np       = data.y.numpy()

            vmag_true = y_np[:, 0]
            vang_true = y_np[:, 1]
            p_true    = y_np[:, 2]
            q_true    = y_np[:, 3]

            # DC predictions --------------------------------------------------
            vmag_dc = np.ones(len(buses))                                      # flat 1.0 p.u.
            vang_dc = net_copy.buses_t.v_ang.loc[snap].reindex(buses).values   # from DC solve
            p_dc    = x_np[:, 3].copy()                                        # given P inputs
            q_dc    = np.zeros(len(buses))                                     # DC: Q=0

            # DC slack P: lossless power balance
            if slack_mask.any():
                p_dc[slack_mask] = -p_true[~slack_mask].sum() / slack_mask.sum()

            # Apply same known-variable overrides as evaluate_gnn_on_test_set -
            vmag_dc[pv_mask]    = x_np[pv_mask,    5]   # PV: Vmag given
            vmag_dc[slack_mask] = x_np[slack_mask, 5]   # Slack: Vmag given
            vang_dc[slack_mask] = x_np[slack_mask, 6]   # Slack: Vang=0 given
            p_dc[pq_mask]       = x_np[pq_mask,    3]   # PQ: P given
            q_dc[pq_mask]       = x_np[pq_mask,    4]   # PQ: Q given
            p_dc[pv_mask]       = x_np[pv_mask,    3]   # PV: P given

            all_vmag_err.extend(np.abs(vmag_dc - vmag_true))  # [STSI 010626]:element-wise errors
            all_vang_err.extend(np.abs(vang_dc - vang_true))
            all_p_err.extend(np.abs(p_dc - p_true))
            all_q_err.extend(np.abs(q_dc - q_true))
            n_snapshots += 1

            # DC line flows (p0 only — q0 is zero in DC) ----------------------
            if not network.lines_t.p0.empty and snap in network.lines_t.p0.index \
                    and snap in net_copy.lines_t.p0.index:
                true_p0 = network.lines_t.p0.loc[snap].values
                dc_p0   = net_copy.lines_t.p0.loc[snap].values
                # sn_mva is cosmetic metadata only (flat-pu convention) — no scaling needed
                all_lflow_err.extend(np.abs(dc_p0 - true_p0))  # [STSI 140926]:removed stale sn_mva division, matches unscaled true_p0

            # DC reactive line flow: naive zero guess (DC has no Q model, mirrors node-level q_dc=0)
            if not network.lines_t.q0.empty and snap in network.lines_t.q0.index:
                true_q0 = network.lines_t.q0.loc[snap].values
                all_lflow_q0_err.extend(np.abs(true_q0))  # guess = 0

            # DC delta_theta from bus angles  # [STSI 010626]:add delta_theta metrics to DC baseline
            dt_errs = _compute_delta_theta_metrics(vang_dc, dataset, net_idx, t_idx, index_map)
            if dt_errs is not None:
                all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

    metrics = {  # [STSI 010626]:element-wise MAE and RMSE (flat list pattern)
        "vmag_mae":    float(np.mean(all_vmag_err)) if all_vmag_err else float("nan"),
        "vmag_rmse":   float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float("nan"),
        "vang_mae":    float(np.mean(all_vang_err)) if all_vang_err else float("nan"),
        "vang_rmse":   float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float("nan"),
        "p_mae":       float(np.mean(all_p_err))    if all_p_err    else float("nan"),
        "p_rmse":      float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float("nan"),
        "q_mae":       float(np.mean(all_q_err))    if all_q_err    else float("nan"),
        "q_rmse":      float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float("nan"),
        "line_flow_mae":     float(np.mean(all_lflow_err)) if all_lflow_err else float("nan"),
        "line_flow_rmse":    float(np.sqrt(np.mean(np.array(all_lflow_err)**2))) if all_lflow_err else float("nan"),
        "line_flow_q0_mae":  float(np.mean(all_lflow_q0_err)) if all_lflow_q0_err else float("nan"),   # naive Q=0 guess
        "line_flow_q0_rmse": float(np.sqrt(np.mean(np.array(all_lflow_q0_err)**2))) if all_lflow_q0_err else float("nan"),
        "delta_theta_mae":   float(np.mean(all_dt_errors)) if all_dt_errors else float("nan"),
        "delta_theta_rmse":  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float("nan"),
        "inference_time_s":               t_solve,
        "inference_time_per_snapshot_ms": 1000.0 * t_solve / max(1, n_snapshots),
        "n_snapshots":  n_snapshots,
    }

    if verbose:
        print(
            f"\n[DC baseline] {n_snapshots} snapshots — "
            f"pure solve time {t_solve:.2f}s "
            f"({metrics['inference_time_per_snapshot_ms']:.3f} ms/snap)"
        )
        for k in ["vmag_mae", "vang_mae", "p_mae", "q_mae", "line_flow_mae"]:
            print(f"  {k:25s}: {metrics[k]:.5f}")

    return metrics


# [COPY of GNN_Powerflow_V2.7_Analysis::predict_network_results_dc]
def predict_network_results_dc(network):  # [STSI 080926]:DC (lpf) solution for single-network comparison plots
    """
    Run PyPSA linear (DC) power flow on one network and return bus/line results
    + prediction masks in the same format as predict_network_results_with_masks,
    so DC can be dropped into plot_single_network_comparisons as another "model".

    DC capabilities:
        vmag: flat 1.0 p.u. (PV/Slack keep known Vmag)
        vang: solved by lpf()
        P:    known inputs; Slack P via lossless power balance
        Q:    naive zero guess (DC has no reactive model)
        line P0/P1: taken directly from lpf() (lossless linear DC flow)
        line Q0/Q1: naive zero guess
    """
    net_copy = network.copy()
    net_copy.lpf()

    snapshots = network.snapshots
    buses = list(network.buses.index)
    lines = list(network.lines.index)

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    for t in snapshots:
        non_slack_p, slack_buses = {}, []
        for bus in buses:
            connected_gens = network.generators[network.generators.bus == bus]
            ctrl = connected_gens.iloc[0]["control"] if len(connected_gens) > 0 else "PQ"
            if ctrl == "Slack":
                slack_buses.append(bus)
            else:
                non_slack_p[bus] = network.buses_t.p.loc[t, bus]
        slack_p_total = -sum(non_slack_p.values())
        n_slack = max(len(slack_buses), 1)

        for bus in buses:
            connected_gens = network.generators[network.generators.bus == bus]
            bus_type = "PQ"
            if len(connected_gens) > 0:
                ctrl = connected_gens.iloc[0]["control"]
                if ctrl in ("Slack", "PV"):
                    bus_type = ctrl
            vang_dc = net_copy.buses_t.v_ang.loc[t, bus]

            if bus_type == "PQ":
                pred_vmag_i, pred_vang_i = 1.0, vang_dc
                pred_p_i = network.buses_t.p.loc[t, bus]
                pred_q_i = network.buses_t.q.loc[t, bus]
                prediction_masks["P"].loc[t, bus]     = False
                prediction_masks["Q"].loc[t, bus]     = False
                prediction_masks["V"].loc[t, bus]     = True
                prediction_masks["Angle"].loc[t, bus] = True
            elif bus_type == "PV":
                pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]
                pred_vang_i = vang_dc
                pred_p_i = network.buses_t.p.loc[t, bus]
                pred_q_i = 0.0  # DC has no Q model
                prediction_masks["P"].loc[t, bus]     = False
                prediction_masks["Q"].loc[t, bus]     = True
                prediction_masks["V"].loc[t, bus]     = False
                prediction_masks["Angle"].loc[t, bus] = True
            else:  # Slack
                pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]
                pred_vang_i = network.buses_t.v_ang.loc[t, bus]  # known = 0
                pred_p_i = slack_p_total / n_slack
                pred_q_i = 0.0  # DC has no Q model
                prediction_masks["P"].loc[t, bus]     = True
                prediction_masks["Q"].loc[t, bus]     = True
                prediction_masks["V"].loc[t, bus]     = False
                prediction_masks["Angle"].loc[t, bus] = False

            bus_results.loc[t, (bus, "P pu")]      = pred_p_i
            bus_results.loc[t, (bus, "Q pu")]      = pred_q_i
            bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(pred_vang_i)

        for line in lines:
            p0 = net_copy.lines_t.p0.loc[t, line] if not net_copy.lines_t.p0.empty else 0.0
            p1 = (net_copy.lines_t.p1.loc[t, line]
                  if hasattr(net_copy.lines_t, "p1") and not net_copy.lines_t.p1.empty
                  else -p0)
            line_results.loc[t, (line, "P0 pu")] = p0
            line_results.loc[t, (line, "P1 pu")] = p1
            line_results.loc[t, (line, "Q0 pu")] = 0.0  # naive Q=0 guess
            line_results.loc[t, (line, "Q1 pu")] = 0.0

    return bus_results, line_results, prediction_masks


# [COPY of GNN_Powerflow_V2.7_Analysis::reconstruct_test_split]
def reconstruct_test_split(networks, seed=42, train_frac=0.70, val_frac=0.15):
    """
    Reproduce the train/val/test split that train_power_flow_gnn uses internally.
    Returns (train_networks, val_networks, test_networks).
    Use this only as a fallback for old runs that don't store test_networks.
    """
    import random as _random
    net_copy = list(networks)
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    _random.shuffle(net_copy)
    n = len(net_copy)
    n_train = int(train_frac * n)
    n_val = int((train_frac + val_frac) * n) - n_train
    train_nets = net_copy[:n_train]
    val_nets = net_copy[n_train : n_train + n_val]
    test_nets = net_copy[n_train + n_val :]
    return train_nets, val_nets, test_nets


# [STSI 141026]:verbatim copies extracted programmatically (task 088); do not edit, originals are ground truth

## Loading helpers

In [ ]:
# ==== CONTINGENCY: LOADING (task 088) ====
# [STSI 141026]:new loading helpers for trained runs, N-1 datasets and their pre-outage base nets
def load_runs_by_key(run_keys, sweep_paths):
    """D6: return {run_key: run_dict} for the user-supplied keys; first occurrence in sweep_paths order wins."""
    if not run_keys:
        raise ValueError("run_keys must be a non-empty list (D6: no default model)")
    found, dups = {}, {}
    for p in sweep_paths:
        for r in load_sweep_results(p):
            k = r.get("run_key")
            if k in run_keys:
                if k in found:
                    dups.setdefault(k, []).append(p)
                else:
                    found[k] = r
    missing = [k for k in run_keys if k not in found]
    if missing:
        raise KeyError(f"run_keys not found in {len(sweep_paths)} file(s): {missing}")
    for k, ps in dups.items():
        print(f"[load_runs_by_key] duplicate {k!r} also in {ps}; using {found[k].get('_source_pkl')}")
    return {k: found[k] for k in run_keys}

BASE_CHECK_FIELDS = ("base_nonconverged_snapshots", "base_sanity_failed_snapshots")   # [STSI 151026]:task 093: task 092 sidecar row fields

def base_failed_snapshots(row):
    """str(snapshot) labels where the pre-outage base failed (inferred non-converged OR N-1 sanity) -> base_nonconverged."""
    return set(row["base_nonconverged_snapshots"]) | set(row["base_sanity_failed_snapshots"])

def drop_base_failed(df, row):
    """Drop records of base-failed snapshots (task 093: class base_nonconverged, never scored, never severe)."""
    bad = base_failed_snapshots(row)
    return df[~df["snapshot"].map(str).isin(bad)] if bad and len(df) else df

def load_n1_dataset(n1_index_path):
    """Returns (nets, header, rows_df); nets[i] is the .nc at rows_df.file_position == i."""
    meta_path = n1_index_path[:-len(".json")] + "_meta.json"
    header, rows = read_n1_sidecar(meta_path)
    missing = [f for f in BASE_CHECK_FIELDS if f not in rows.columns]   # [STSI 151026]:task 093: no fallback for old sidecars
    if missing or not all(isinstance(v, list) for f in BASE_CHECK_FIELDS for v in rows[f]):
        raise ValueError(f"{meta_path}: sidecar rows lack base-check field(s) {missing or list(BASE_CHECK_FIELDS)}; "
                         "regenerate the N-1 dataset with DataGen task 092+ (no fallback)")
    nets = load_dataset_list(n1_index_path)
    solved = rows[rows.status == "solved"]
    assert list(solved.file_position.astype(int)) == list(range(len(nets))), "sidecar/index misaligned"
    return nets, header, rows

def load_source_base_nets(rows_df):
    """{(source_index, source_position): pre-outage base net} for every base case referenced in the sidecar."""
    out = {}
    for src, grp in rows_df.groupby("source_index"):
        for pos, _, net in load_dataset_subset(src, sorted(set(int(p) for p in grp.source_position))):
            out[(src, pos)] = net
    return out

## Violation assessment

In [ ]:
# ==== CONTINGENCY: VIOLATION ASSESSMENT (task 089) ====
# [STSI 141026]:per-element violation labels (truth / DC / GNN runs) on solved N-1 cases + reference timing
V_BAND_VIOLATION      = (0.90, 1.10)   # D12 'violation' band (pu)
V_BAND_SEVERE         = (0.85, 1.15)   # D12 'severe' band (pu)
THERMAL_SEVERE_FACTOR = 1.2            # D12b: severe if loading > factor


def bus_types(net):
    """Slack/PV/PQ per bus from the first generator's control (same rule as predict_network_results_dc)."""
    t = pd.Series("PQ", index=net.buses.index)
    for bus, ctrl in net.generators.groupby("bus")["control"].first().items():
        if ctrl in ("Slack", "PV"):
            t[bus] = ctrl
    return t


def truth_state(net):
    b, l = net.buses.index, net.lines.index # bus and line indices
    return dict(vmag=net.buses_t.v_mag_pu[b], p0=net.lines_t.p0[l], q0=net.lines_t.q0[l],
                p1=net.lines_t.p1[l], q1=net.lines_t.q1[l]) # extract the truth state for the given bus and line indices based on the network's time series data


def state_from_prediction(bus_results, line_results, net):
    b, l = net.buses.index, net.lines.index
    lx = lambda f: line_results.xs(f, axis=1, level=1)[l].astype(float)
    return dict(vmag=bus_results.xs("V pu", axis=1, level=1)[b].astype(float),
                p0=lx("P0 pu"), q0=lx("Q0 pu"), p1=lx("P1 pu"), q1=lx("Q1 pu"))


def predict_state_gnn(run, net):
    hp = run.get("hparams") or {}   # same parameter resolution as Analysis check_hparam_results
    out = predict_network_results_with_masks( # predict the network results using the GNN model with the given network and hyperparameters
        run["model"], net, use_edge_features=True, use_pnom_share=None, 
        ptdf_params=run.get("_ptdf_params") or hp.get("ptdf_params", None), # get the PTDF parameters from the run configuration or hyperparameters
        max_buses=run.get("_max_buses") or hp.get("max_buses", 0), # get the maximum number of buses from the run configuration or hyperparameters
        ptdf_offset=run.get("_ptdf_offset") or hp.get("ptdf_offset", 0)) # get the PTDF offset from the run configuration or hyperparameters
    return state_from_prediction(out[0], out[1], net)   # returns (bus_results, line_results, masks, dt_dict)


def predict_state_dc(net):
    out = predict_network_results_dc(net)
    return state_from_prediction(out[0], out[1], net)


def _viol_long_frame(index, columns, key, cols):
    """Long format: one row per (snapshot, element); every array in `cols` is [snapshot x element]."""
    n_t, n_e = len(index), len(columns)
    df = pd.DataFrame({"snapshot": np.repeat(np.asarray(index), n_e), key: np.tile(np.asarray(columns), n_t)})
    for name, a in cols.items():
        df[name] = np.asarray(a).reshape(n_t * n_e)
    return df


def assess_bus_violations(vmag, pre_vmag, btype, v_band=V_BAND_VIOLATION, v_band_severe=V_BAND_SEVERE):
    """D12 bus voltage labels (strict inequalities: band edges are not violations); pre-outage aligned on snapshot x bus."""
    vmin, vmax = v_band
    smin, smax = v_band_severe
    v = vmag.to_numpy(dtype=float)
    pv = pre_vmag.loc[vmag.index, vmag.columns].to_numpy(dtype=float)
    under, over = v < vmin, v > vmax
    sev_u, sev_o = v < smin, v > smax
    pre_u, pre_o = pv < vmin, pv > vmax
    ind_u, ind_o = under & ~pre_u, over & ~pre_o
    bt = np.broadcast_to(pd.Series(btype).loc[vmag.columns].to_numpy(dtype=object), v.shape)
    return _viol_long_frame(vmag.index, vmag.columns, "bus", dict(
        bus_type=bt, vmag=v, under_v=under, over_v=over, violation=under | over,
        severe_under=sev_u, severe_over=sev_o, severe=sev_u | sev_o,
        margin_v=np.minimum(v - vmin, vmax - v), pre_under=pre_u, pre_over=pre_o,
        induced_under=ind_u, induced_over=ind_o, contingency_induced=ind_u | ind_o))


def assess_line_violations(state, pre_state, s_limit, limit_source, severe_factor=THERMAL_SEVERE_FACTOR):
    """D12b/D14/D11b thermal labels at both ends; NaN limit -> rated=False and every thermal label False."""
    idx, cols = state["p0"].index, state["p0"].columns
    g = lambda d, f: d[f].loc[idx, cols].to_numpy(dtype=float)
    s0, s1 = np.hypot(g(state, "p0"), g(state, "q0")), np.hypot(g(state, "p1"), g(state, "q1"))
    pre_s = np.maximum(np.hypot(g(pre_state, "p0"), g(pre_state, "q0")),
                       np.hypot(g(pre_state, "p1"), g(pre_state, "q1")))
    lim = np.broadcast_to(pd.Series(s_limit).loc[cols].to_numpy(dtype=float), s0.shape)
    src = np.broadcast_to(pd.Series(limit_source).loc[cols].to_numpy(dtype=object), s0.shape)
    rated = ~np.isnan(lim)
    with np.errstate(divide="ignore", invalid="ignore"):
        load0, load1, pre_loading = s0 / lim, s1 / lim, pre_s / lim
        loading = np.maximum(load0, load1)
        overload = rated & (loading > 1.0)
        severe = rated & (loading > severe_factor)
        pre_overload = rated & (pre_loading > 1.0)
    return _viol_long_frame(idx, cols, "line", dict(
        s0=s0, s1=s1, s_limit_pu=lim, limit_source=src, rated=rated,
        loading0=load0, loading1=load1, loading=loading, overload=overload, severe_overload=severe,
        pre_loading=pre_loading, pre_overload=pre_overload, induced_overload=overload & ~pre_overload))


_CASE_COLS = ("case_id", "system", "outage_name", "source_index", "source_position")


def assess_case(state, net, row, pre_net, v_band=V_BAND_VIOLATION, v_band_severe=V_BAND_SEVERE,
                severe_factor=THERMAL_SEVERE_FACTOR):
    """(bus_df, line_df) for one N-1 case. Reads only `state` and `pre_net` solutions, never net.buses_t/lines_t."""
    pre = truth_state(pre_net)
    bus_df = assess_bus_violations(state["vmag"], pre["vmag"], bus_types(net), v_band, v_band_severe)
    pre_lines = {f: pre[f][net.lines.index] for f in ("p0", "q0", "p1", "q1")}
    line_df = assess_line_violations(state, pre_lines, net.lines["s_limit_pu"], net.lines["limit_source"],
                                     severe_factor)
    for df in (bus_df, line_df):
        for i, k in enumerate(_CASE_COLS):
            df.insert(i, k, row[k])
    return bus_df, line_df


def run_violation_assessment(n1_nets, rows_df, base_nets, runs, include_dc=True, v_band=V_BAND_VIOLATION,
                             v_band_severe=V_BAND_SEVERE, severe_factor=THERMAL_SEVERE_FACTOR):
    """{predictor: (bus_df, line_df)} over all SOLVED cases; predictors = 'truth', 'dc', *run_keys.
    result["_timing_s"][predictor] = summed wall time (s) of building that predictor's states."""
    preds = {"truth": truth_state} # initialize the dictionary of predictors with the truth state based on the network's time series data and the solution from the power flow analysis done when network is generated
    if include_dc:
        preds["dc"] = predict_state_dc # wrapper that uses the DC power flow approximation to predict the state based on the function "predict_network_results_dc" with the given network
    for k, r in (runs or {}).items(): # k is the run key, r is the corresponding run configuration
        if k in preds:
            raise ValueError(f"run key {k!r} collides with a built-in predictor name") # guard to prevent overwriting existing predictors
        preds[k] = lambda net, _r=r: predict_state_gnn(_r, net) # add a new predictor using the GNN model for the given run configuration using the predict_network_results_with_masks function based on the extra run configuration
    parts = {p: ([], []) for p in preds} # initialize empty lists for bus and line dataframes for each predictor
    timing = {p: 0.0 for p in preds} # initialize timing dictionary to keep track of the wall time for each predictor
    solved = rows_df[rows_df.status == "solved"] # filter the rows to only include solved cases
    for _, row in tqdm(solved.iterrows(), total=len(solved), desc="N-1 violation assessment"): # iterate over each solved case
        net = n1_nets[int(row["file_position"])] # get the corresponding network for the current row
        pre_net = base_nets[(row["source_index"], int(row["source_position"]))] # get the pre-contingency network
        for p, fn in preds.items(): # iterate over each predictor
            t0 = time.perf_counter() # start the timer
            state = fn(net) # predict the state using the current predictor
            timing[p] += time.perf_counter() - t0 # update the timing for the current predictor
            b, l = assess_case(state, net, row, pre_net, v_band, v_band_severe, severe_factor) # assess violations for the current case
            parts[p][0].append(drop_base_failed(b, row))   # [STSI 151026]:task 093 D-C: base-failed snapshots never scored
            parts[p][1].append(drop_base_failed(l, row)) # append the assessed line violations after dropping base-failed snapshots
    result = {p: tuple(pd.concat(x, ignore_index=True).assign(predictor=p) if x else pd.DataFrame() for x in bl) # combine the bus and line dataframes for each predictor into a tuple, adding the predictor column
              for p, bl in parts.items()}
    result["_timing_s"] = timing
    return result


def run_reference_timing(n1_nets, n_subset=20, seed=0):
    """D4(c): existing aggregate evaluators (copied build_ref_baselines) on a seeded random subset of solved nets."""
    k = min(n_subset, len(n1_nets))
    subset = [n1_nets[i] for i in sorted(random.Random(seed).sample(range(len(n1_nets)), k))]
    return build_ref_baselines(None, n_sample=None, test_networks=subset)

## No-solution outages (rebuilt graphs)

In [ ]:
# ==== CONTINGENCY: NO-SOLUTION OUTAGES (task 091) ====
# [STSI 141026]:predictor records (DC / GNN runs, no truth) for non_converged / sanity_rejected outages on rebuilt graphs
NO_SOLUTION_STATUSES = ("non_converged", "sanity_rejected")   # D10b(iii), user 2026-10-14


def decode_line_limits(header):
    """Sidecar header line_limits (task 087) -> (real {system: {name: (v, src)}}, synth {(system, name): v})."""
    lim = header["line_limits"]
    real = {s: {n: (float(v), src) for n, (v, src) in d.items()} for s, d in lim["real"].items()}
    synth = {(s, n): float(v) for s, n, v in lim["synthetic"]}
    return real, synth


def rebuild_post_outage_net(base_net, row, real, synth):
    """D10b(i)(c): source base net minus the outaged line, limits re-attached; NO re-solve.
    buses_t/lines_t keep the PRE-outage solution -> valid predictor input (setpoints), never truth."""
    if row.outage_component != "Line":
        raise ValueError(f"only Line outages are supported (D2), got {row.outage_component!r}")
    net = base_net.copy()
    net.remove("Line", row.outage_name)
    return attach_line_limits(net, row.system, real.get(row.system, {}), synth)


def run_no_solution_assessment(rows_df, base_nets, header, runs, include_dc=True, statuses=NO_SOLUTION_STATUSES,
                               v_band=V_BAND_VIOLATION, v_band_severe=V_BAND_SEVERE,
                               severe_factor=THERMAL_SEVERE_FACTOR):
    """{predictor: (bus_df, line_df), '_timing_s': {...}} over no-solution outages, all base-ok snapshots (D10b(ii), task 093).
    Same schema as run_violation_assessment + 'status'; predictors = 'dc', *run_keys; never 'truth'."""
    real, synth = decode_line_limits(header)
    sel = rows_df[rows_df.status.isin(statuses)]
    if sel.empty:
        return {"_timing_s": {}}
    preds = {}
    if include_dc:
        preds["dc"] = predict_state_dc
    for k, r in (runs or {}).items():
        if k in preds or k == "truth":
            raise ValueError(f"run key {k!r} collides with a built-in predictor name")
        preds[k] = lambda net, _r=r: predict_state_gnn(_r, net)
    parts = {p: ([], []) for p in preds}
    timing = {p: 0.0 for p in preds}
    for _, row in tqdm(sel.iterrows(), total=len(sel), desc="N-1 no-solution assessment"):
        base = base_nets[(row["source_index"], int(row["source_position"]))]
        net = rebuild_post_outage_net(base, row, real, synth)
        for p, fn in preds.items():
            t0 = time.perf_counter()
            state = fn(net)
            timing[p] += time.perf_counter() - t0
            b, l = assess_case(state, net, row, base, v_band=v_band, v_band_severe=v_band_severe,
                               severe_factor=severe_factor)
            b["status"] = row["status"]
            l["status"] = row["status"]
            parts[p][0].append(drop_base_failed(b, row))   # [STSI 151026]:task 093 D-A: base-failed snapshots = base_nonconverged, never severe
            parts[p][1].append(drop_base_failed(l, row))
    result = {p: tuple(pd.concat(x, ignore_index=True).assign(predictor=p) if x else pd.DataFrame() for x in bl)
              for p, bl in parts.items()}
    result["_timing_s"] = timing
    return result

## Metrics & report

In [ ]:
# ==== CONTINGENCY: METRICS & REPORT (task 090) ====
# [STSI 141026]:D5 metrics (co-headlines: case-level recall + element-level F1), outage summary, N-1-vs-base error, cost, report
import warnings

VIOLATION_LEVELS = ("violation", "severe", "contingency_induced")
CASE_SCOPES      = ("voltage", "thermal", "any")
# (element, violation_type) -> {level: per-element label column of task 089}
LABEL_SETS = {
    ("bus", "under_v"):   {"violation": "under_v",  "severe": "severe_under",    "contingency_induced": "induced_under"},
    ("bus", "over_v"):    {"violation": "over_v",   "severe": "severe_over",     "contingency_induced": "induced_over"},
    ("line", "overload"): {"violation": "overload", "severe": "severe_overload", "contingency_induced": "induced_overload"},
}
_SCOPE_LABELS = {"voltage": {"violation": "violation", "severe": "severe", "contingency_induced": "contingency_induced"},
                 "thermal": LABEL_SETS[("line", "overload")]}
_VALUE_COLS = {"bus": ("vmag",), "line": ("loading", "rated")}
_CASE_KEY = ["case_id", "snapshot"]
_CASE_FLAGS = [f"any_{lv}_{sc}" for lv in VIOLATION_LEVELS for sc in CASE_SCOPES]
_COUNT_COLS = ["tp", "fp", "fn", "tn", "n", "precision", "recall", "miss_rate", "f1", "specificity"]
_REG_COLS = ["vmag_mae", "vang_mae", "delta_theta_mae", "p_mae", "q_mae", "line_flow_mae", "line_flow_q0_mae"]
NO_SOLUTION_CAVEAT = ("no-solution flags include PV/slack setpoint voltages (D3 caveat); "
                      "level 'violation' can be near-trivially positive")
D3_CAVEAT = ("D3 caveat: post-outage dispatch is single slack with no Q limits, so PV/slack bus voltages equal "
             "their setpoints (voltage violations can only appear at PQ buses)")


def classification_counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(np.asarray(y_true, bool), np.asarray(y_pred, bool), labels=[False, True]).ravel() # confusion matrix counts: true negatives, false positives, false negatives, true positives
    div = lambda a, b: a / b if b else float("nan")
    prec, rec = div(tp, tp + fp), div(tp, tp + fn) 
    return dict(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn), n=int(tp + fp + fn + tn),
                precision=prec, recall=rec, miss_rate=div(fn, tp + fn), # precision here means the proportion of predicted positives that are correct; recall here means the proportion of actual positives correctly identified by the predictor, miss_rate is the proportion of actual positives missed by the predictor
                f1=div(2 * prec * rec, prec + rec) if tp else (0.0 if (fp + fn) else float("nan")), # F1 score: harmonic mean of precision and recall, 0 if no true positives but there are false positives/negatives, NaN if no predictions at all
                specificity=div(tn, tn + fp))


def _predictors(assess):
    return [(p, recs) for p, recs in (assess or {}).items() if p not in ("truth", "_timing_s")]


def _systems(*dfs):
    found = set()
    for d in dfs:
        if d is not None and len(d) and "system" in d.columns:
            found |= set(d["system"])
    return ["ALL", *sorted(found)]


def _in_system(df, system):
    return df if system == "ALL" else df[df["system"] == system]


def _merge_truth(truth_df, pred_df, key, cols):
    """Inner one-to-one merge on (case_id, snapshot, bus|line); every column in `cols` gets suffix _t / _p."""
    on = [*_CASE_KEY, key]
    return truth_df[[*on, "system", *cols]].merge(pred_df[[*on, *cols]], on=on, how="inner",
                                                    suffixes=("_t", "_p"), validate="one_to_one")


def _element_pairs(assess):
    """Yields (predictor, element, violation_type, levels, merged); line rows restricted to truth.rated."""
    t_bus, t_line = assess["truth"]
    for p, (p_bus, p_line) in _predictors(assess): # iterate over all predictors excluding 'truth' and '_timing_s'
        for (el, vt), levels in LABEL_SETS.items(): # el is the element type ('bus' or 'line'), vt is the violation type, levels is the dictionary of level names to column names
            t, pr = (t_bus, p_bus) if el == "bus" else (t_line, p_line) # select the appropriate truth and predicted dataframes based on the element type
            if not len(t) or not len(pr):
                continue
            m = _merge_truth(t, pr, el, [*levels.values(), *_VALUE_COLS[el]]) # merge the truth and predicted dataframes on the key columns and include the relevant level and value columns
            if el == "line":
                m = m[m["rated_t"].astype(bool)]
            yield p, el, vt, levels, m


def element_level_metrics(assess, by_system=True):
    """Rows [predictor, system ('ALL' + each), element, violation_type, level, **counts]. Co-headline 2 (f1)."""
    rows = []
    for p, el, vt, levels, m in _element_pairs(assess):
        for system in (_systems(m) if by_system else ["ALL"]):
            s = _in_system(m, system)
            if not len(s):
                continue
            for level, c in levels.items():
                rows.append(dict(predictor=p, system=system, element=el, violation_type=vt, level=level,
                                 **classification_counts(s[f"{c}_t"], s[f"{c}_p"])))
    return pd.DataFrame(rows, columns=["predictor", "system", "element", "violation_type", "level", *_COUNT_COLS])


def case_snapshot_labels(bus_df, line_df, v_band=V_BAND_VIOLATION):
    """One row per (case_id, snapshot): system[, status], any_<level>_<scope>, max_loading, sum_overload (rated
    lines), min_v, max_v_dev (= max(0, vmin-v, v-vmax) over buses)."""
    tail = ["max_loading", "sum_overload", "min_v", "max_v_dev"] # additional computed metrics for each case-snapshot
    meta = [d[[*_CASE_KEY, *[c for c in ("system", "status") if c in d.columns]]] for d in (bus_df, line_df) if len(d)] # extract relevant metadata from bus and line dataframes
    if not meta:
        return pd.DataFrame(columns=[*_CASE_KEY, "system", *_CASE_FLAGS, *tail])
    out = pd.concat(meta, ignore_index=True).groupby(_CASE_KEY, sort=True).first()
    vmin, vmax = v_band
    if len(bus_df):
        v = bus_df["vmag"].astype(float)
        b = bus_df[_CASE_KEY].assign(min_v=v, max_v_dev=np.maximum(0.0, np.maximum(vmin - v, v - vmax)),
                                     **{f"any_{lv}_voltage": bus_df[c].astype(bool)
                                        for lv, c in _SCOPE_LABELS["voltage"].items()})
        out = out.join(b.groupby(_CASE_KEY).agg({"min_v": "min", "max_v_dev": "max",
                                                 **{f"any_{lv}_voltage": "any" for lv in VIOLATION_LEVELS}}))
    if len(line_df):
        ld = line_df[line_df["rated"].astype(bool).to_numpy()]
        loading = ld["loading"].astype(float)
        l = ld[_CASE_KEY].assign(max_loading=loading, sum_overload=np.maximum(0.0, loading - 1.0),
                                 **{f"any_{lv}_thermal": ld[c].astype(bool)
                                    for lv, c in _SCOPE_LABELS["thermal"].items()})
        out = out.join(l.groupby(_CASE_KEY).agg({"max_loading": "max", "sum_overload": "sum",
                                                 **{f"any_{lv}_thermal": "any" for lv in VIOLATION_LEVELS}}))
    for sc in ("voltage", "thermal"):
        for lv in VIOLATION_LEVELS:
            c = f"any_{lv}_{sc}"
            out[c] = out[c].eq(True) if c in out.columns else False   # missing (no rated line / no bus) -> False
    for lv in VIOLATION_LEVELS:
        out[f"any_{lv}_any"] = out[f"any_{lv}_voltage"] | out[f"any_{lv}_thermal"]
    for c in tail:
        out[c] = out[c].astype(float) if c in out.columns else float("nan")
    out["sum_overload"] = out["sum_overload"].fillna(0.0)
    keep = [c for c in ("system", "status") if c in out.columns]
    return out.reset_index()[[*_CASE_KEY, *keep, *_CASE_FLAGS, *tail]]


def case_level_metrics(assess, no_solution_assess=None):
    """Rows [predictor, system, level, scope, **counts] over SOLVED case-snapshots (scopes voltage/thermal/any).
    With no_solution_assess: plus scope 'any+no_solution' (levels violation/severe) where every no-solution
    case-snapshot is truth-positive (D10b(ii)). Co-headline 1."""
    tl = case_snapshot_labels(*assess["truth"]) # truth labels for all case snapshots
    rows = []
    for p, recs in _predictors(assess): # iterate over each predictor and its corresponding records
        m = tl[[*_CASE_KEY, "system", *_CASE_FLAGS]].merge( # merge truth labels with predictor records on case key
            case_snapshot_labels(*recs)[[*_CASE_KEY, *_CASE_FLAGS]], on=_CASE_KEY, how="inner", # inner join to keep only matching case snapshots
            suffixes=("_t", "_p"), validate="one_to_one") # ensure that each case snapshot has exactly one truth and one prediction record
        ns_lab = None # no-solution labels for the current predictor, if available
        if no_solution_assess is not None: # check if no-solution assessment is provided for the current predictor
            ns_lab = (case_snapshot_labels(*no_solution_assess[p]) if p in no_solution_assess
                      else pd.DataFrame(columns=[*_CASE_KEY, "system", *_CASE_FLAGS])) # get no-solution labels for the current predictor, or an empty DataFrame if not available
        for system in _systems(m, ns_lab): # iterate over each system present in the merged truth-prediction DataFrame and no-solution labels
            s = _in_system(m, system) # get all merged truth-prediction records for the current system
            if len(s): # proceed only if there are records for the current system
                for lv in VIOLATION_LEVELS:
                    for sc in CASE_SCOPES:
                        c = f"any_{lv}_{sc}"
                        rows.append(dict(predictor=p, system=system, level=lv, scope=sc,
                                         **classification_counts(s[f"{c}_t"], s[f"{c}_p"])))
            if ns_lab is None:
                continue
            n = _in_system(ns_lab, system)
            for lv in ("violation", "severe"):
                c = f"any_{lv}_any"
                y_true = np.concatenate([s[f"{c}_t"].to_numpy(dtype=bool), np.ones(len(n), dtype=bool)])
                y_pred = np.concatenate([s[f"{c}_p"].to_numpy(dtype=bool), n[c].to_numpy(dtype=bool)])
                rows.append(dict(predictor=p, system=system, level=lv, scope="any+no_solution",
                                 **classification_counts(y_true, y_pred)))
    return pd.DataFrame(rows, columns=["predictor", "system", "level", "scope", *_COUNT_COLS])


def no_solution_flag_table(no_solution_assess):
    """Per predictor x system x status x level: how many no-solution case-snapshots / outages the predictor flags.
    flag_rate_snapshot = recall on the no-solution subset; flag_rate_outage = outage flagged in >= 1 snapshot."""
    rows = []
    for p, recs in _predictors(no_solution_assess):
        lab = case_snapshot_labels(*recs)
        for system in _systems(lab):
            s1 = _in_system(lab, system)
            statuses = sorted(set(s1["status"])) if "status" in s1.columns else []
            for status in ["ALL", *statuses]:
                s = s1 if status == "ALL" else s1[s1["status"] == status]
                if not len(s):
                    continue
                for lv in ("violation", "severe"):
                    f = s[f"any_{lv}_any"].astype(bool)
                    per_outage = f.groupby(s["case_id"]).any()
                    rows.append(dict(predictor=p, system=system, status=status, level=lv,
                                     n_outages=int(len(per_outage)), n_case_snapshots=int(len(s)),
                                     n_flagged_case_snapshots=int(f.sum()), flag_rate_snapshot=float(f.mean()),
                                     n_outages_flagged_any_snapshot=int(per_outage.sum()),
                                     flag_rate_outage=float(per_outage.mean())))
    return pd.DataFrame(rows, columns=["predictor", "system", "status", "level", "n_outages", "n_case_snapshots",
                                       "n_flagged_case_snapshots", "flag_rate_snapshot",
                                       "n_outages_flagged_any_snapshot", "flag_rate_outage"])


def outage_summary(rows_df, truth_case_labels, base_nets, no_solution_statuses=NO_SOLUTION_STATUSES):
    """Per system (+ALL): n_outages, n_islanding (bridge_skipped), n_non_converged, n_sanity_rejected, n_solved,
    case-snapshot counts: solved, islanding (= n_bridge x n_snapshots), no_solution_severe, and shares of
    any_violation / any_severe where severe INCLUDES no-solution case-snapshots (D10b: whole outage counted as 'severe')
    EXCEPT base-failed snapshots, reported as cs_base_nonconverged_no_solution / _solved (task 093)."""
    n_snap = pd.Series([len(base_nets[(s, int(p))].snapshots) # number of snapshots for each base network corresponding to the source index and position in the rows dataframe
                        for s, p in zip(rows_df["source_index"], rows_df["source_position"])], # create a list of the number of snapshots for each base network
                       index=rows_df.index, dtype=float)
    n_bad = pd.Series([len(base_failed_snapshots(r) & {str(t) for t in base_nets[(r["source_index"], int(r["source_position"]))].snapshots}) # number of base-failed snapshots for each row corresponding to the source index and position in the rows dataframe
                       for _, r in rows_df.iterrows()], index=rows_df.index, dtype=float)   # [STSI 151026]:task 093: base_nonconverged
    div = lambda a, b: a / b if b else float("nan") # safe division function that returns NaN if the denominator is zero
    rows = []
    for system in _systems(rows_df, truth_case_labels):# iterate over each unique system in the rows dataframe and truth case labels
        r = _in_system(rows_df, system) # get the subset of rows corresponding to the current system
        st, ns_, nb_ = r["status"], n_snap.loc[r.index], n_bad.loc[r.index] # extract the status, number of snapshots, and number of base-failed snapshots for the current system's rows
        t = _in_system(truth_case_labels, system) # get the subset of truth case labels corresponding to the current system
        bridge, no_sol = (st == "bridge_skipped").to_numpy(), st.isin(no_solution_statuses).to_numpy() # boolean arrays indicating bridge-skipped and no-solution statuses for the current system's rows
        cs_solved, cs_isl, cs_ns = int(len(t)), int(ns_[bridge].sum()), int((ns_ - nb_)[no_sol].sum())   # [STSI 151026]:task 093 D-A: compute case-snapshot counts for solved, islanding, and no-solution severe scenarios 
        n_v, n_s = int(t["any_violation_any"].astype(bool).sum()), int(t["any_severe_any"].astype(bool).sum()) # n_v is the number of any-violation snapshots, n_s is the number of any-severe snapshots
        rows.append(dict(system=system, n_outages=int(len(r)), n_islanding=int(bridge.sum()), # append the computed metrics for the current system to the rows list
                         n_non_converged=int((st == "non_converged").sum()), # number of non-converged outages for the current system (where the contingency did not converge successfully, base-case non-converged included?)
                         n_sanity_rejected=int((st == "sanity_rejected").sum()), n_solved=int((st == "solved").sum()), # number of sanity-rejected and solved outages for the current system
                         cs_solved=cs_solved, cs_islanding=cs_isl, cs_no_solution_severe=cs_ns, # case-snapshot counts for solved, islanding, and no-solution severe scenarios
                         cs_base_nonconverged_no_solution=int(nb_[no_sol].sum()), # number of base-nonconverged snapshots for no-solution cases
                         cs_base_nonconverged_solved=int(nb_[(st == "solved").to_numpy()].sum()), # number of base-nonconverged snapshots for solved cases
                         share_any_violation_solved=div(n_v, cs_solved), share_any_severe_solved=div(n_s, cs_solved), # share of any violation and any severe cases among solved cases
                         share_any_violation_incl_no_solution=div(n_v + cs_ns, cs_solved + cs_ns), # share of any violation including no-solution severe cases
                         share_any_severe_incl_no_solution=div(n_s + cs_ns, cs_solved + cs_ns))) # share of any severe including no-solution severe cases
    return pd.DataFrame(rows)


def case_severity(bus_df, line_df, v_band=V_BAND_VIOLATION):
    """Per case_id, max over snapshots: thermal = max loading (rated lines), voltage = max band deviation."""
    lab = case_snapshot_labels(bus_df, line_df, v_band)
    return lab.groupby("case_id").agg(thermal=("max_loading", "max"), voltage=("max_v_dev", "max"))


def _top_k_recall(true_sev, pred_sev, k):
    if len(true_sev) < k:
        return float("nan")
    top = lambda s: set(s.sort_values(ascending=False, kind="mergesort").index[:k])
    return len(top(true_sev) & top(pred_sev)) / k


def severity_ranking(assess, k_list=(5, 10, 20), v_band=V_BAND_VIOLATION):
    """Spearman rho and top-k recall of predicted vs true case severity over all solved cases."""
    ts = case_severity(*assess["truth"], v_band=v_band)
    rows = []
    for p, recs in _predictors(assess):
        ps = case_severity(*recs, v_band=v_band)
        for sev in ("thermal", "voltage"):
            j = pd.concat([ts[sev].rename("t"), ps[sev].rename("p")], axis=1, join="inner").dropna()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")   # constant input -> NaN rho
                rho = float(spearmanr(j["t"], j["p"])[0]) if len(j) >= 2 else float("nan")
            row = dict(predictor=p, severity=sev, n_cases=int(len(j)), spearman_rho=rho)
            for k in k_list:
                row[f"top{k}_recall"] = _top_k_recall(j["t"], j["p"], k)
            rows.append(row)
    return pd.DataFrame(rows, columns=["predictor", "severity", "n_cases", "spearman_rho",
                                       *[f"top{k}_recall" for k in k_list]])


def pr_auc_table(assess, v_band=V_BAND_VIOLATION):
    """Threshold-free average precision per (violation_type, level); NaN if truth has no positives."""
    
    vmin, vmax = v_band
    scores = { # dictionary mapping violation types to scoring functions to be applied to the merged dataframe `m` for each element type and violation type
        "under_v": lambda m: vmin - m["vmag_p"].astype(float), # score for under voltage violations by subtracting the predicted voltage from the minimum voltage limit
              "over_v": lambda m: m["vmag_p"].astype(float) - vmax, # score for over voltage violations by subtracting the maximum voltage limit from the predicted voltage
              "overload": lambda m: m["loading_p"].astype(float)} # score for overload violations using the predicted line loading
    rows = []
    for p, el, vt, levels, m in _element_pairs(assess):
        sc = scores[vt](m).to_numpy()
        ok = np.isfinite(sc)
        for level, c in levels.items():
            y = m[f"{c}_t"].to_numpy(dtype=bool)[ok]
            ap = float(average_precision_score(y, sc[ok])) if y.any() else float("nan") # ap is calculated as the area under the precision-recall curve 
            rows.append(dict(predictor=p, element=el, violation_type=vt, level=level,
                             n=int(ok.sum()), n_pos=int(y.sum()), pr_auc=ap)) # pr_auc here is the average precision under the precision-recall curve
    return pd.DataFrame(rows, columns=["predictor", "element", "violation_type", "level", "n", "n_pos", "pr_auc"])


def near_limit_mae(assess, tol=0.05, v_band=V_BAND_VIOLATION):
    """MAE on elements near a limit (relative tol): rated lines |loading_true-1| <= tol (MAE in percentage points);
    buses |v_true-vmin| <= tol*vmin or |v_true-vmax| <= tol*vmax (MAE in pu)."""
    vmin, vmax = v_band
    t_bus, t_line = assess["truth"]
    rows = []
    for p, (p_bus, p_line) in _predictors(assess):
        if len(t_bus) and len(p_bus):
            m = _merge_truth(t_bus, p_bus, "bus", ["vmag"])
            v = m["vmag_t"].astype(float)
            sel = ((v - vmin).abs() <= tol * vmin) | ((v - vmax).abs() <= tol * vmax)
            err = (m.loc[sel, "vmag_p"].astype(float) - v[sel]).abs()
            rows.append(dict(predictor=p, element="bus", quantity="vmag", unit="pu", n=int(sel.sum()),
                             mae=float(err.mean()) if len(err) else float("nan")))
        if len(t_line) and len(p_line):
            m = _merge_truth(t_line, p_line, "line", ["loading", "rated"])
            m = m[m["rated_t"].astype(bool)]
            lt = m["loading_t"].astype(float)
            sel = (lt - 1.0).abs() <= tol
            err = 100.0 * (m.loc[sel, "loading_p"].astype(float) - lt[sel]).abs()
            rows.append(dict(predictor=p, element="line", quantity="loading", unit="pct_points", n=int(sel.sum()),
                             mae=float(err.mean()) if len(err) else float("nan")))
    return pd.DataFrame(rows, columns=["predictor", "element", "quantity", "unit", "n", "mae"])


def n1_vs_base_regression(runs, n1_nets, base_net_list):
    """GNN regression error on the pre-outage base nets vs the N-1 nets (copied evaluate_gnn_on_test_set).
    Returns (df [run_key, set, *_REG_COLS], raw {(run_key, set): metrics dict}); missing keys -> NaN."""
    rows, raw = [], {}
    for k, run in (runs or {}).items():
        hp = run.get("hparams") or {}   # same parameter resolution as predict_state_gnn (task 089)
        kw = dict(use_edge_features=True, use_pnom_share=None,
                  ptdf_params=run.get("_ptdf_params") or hp.get("ptdf_params", None),
                  max_buses=run.get("_max_buses") or hp.get("max_buses", 0),
                  ptdf_offset=run.get("_ptdf_offset") or hp.get("ptdf_offset", 0), verbose=False)
        for set_name, nets in (("base", base_net_list), ("n1", n1_nets)):
            met = evaluate_gnn_on_test_set(run["model"], list(nets), **kw)
            raw[(k, set_name)] = met
            rows.append(dict(run_key=k, set=set_name, **{c: float(met.get(c, float("nan"))) for c in _REG_COLS}))
    return pd.DataFrame(rows, columns=["run_key", "set", *_REG_COLS]), raw


def restrict_nets_to_base_ok(n1_nets, rows_df, base_nets):   # [STSI 151026]:task 093 D-C
    """(n1_list, base_list) for n1_vs_base_regression without base-failed snapshots (task 093 D-C);
    the original net objects are returned unchanged when no snapshot failed."""
    bad = {(r["source_index"], int(r["source_position"])): base_failed_snapshots(r) for _, r in rows_df.iterrows()}
    if not any(bad.values()):
        return list(n1_nets), list(base_nets.values())

    def keep(net, b):
        kept = [t for t in net.snapshots if str(t) not in b]
        return net if len(kept) == len(net.snapshots) else (net.copy(snapshots=kept) if kept else None)

    solved = rows_df[rows_df.status == "solved"]
    n1 = [keep(n1_nets[int(r["file_position"])], bad[(r["source_index"], int(r["source_position"]))])
          for _, r in solved.iterrows()]
    base = [keep(net, bad.get(k, set())) for k, net in base_nets.items()]
    return [n for n in n1 if n is not None], [n for n in base if n is not None]


def cost_table(raw_n1, ref_single, ref_batch, timing_s, n_cases, n_snapshots):
    """ms per snapshot and per contingency (= per-snapshot ms x n_snapshots) for GNN runs, reference solvers and
    the assessment pipeline (wall time per solved case, incl. pandas)."""
    rows = []

    def add(method, ms, src):
        ms = float(ms) if ms is not None else float("nan")
        rows.append(dict(method=method, ms_per_snapshot=ms, ms_per_contingency=ms * n_snapshots, source_key=src))

    for k, m in (raw_n1 or {}).items():
        add(f"GNN {k} (batched)", m.get("batch_solve_time_ms"), "batch_solve_time_ms")
        add(f"GNN {k} (single)", m.get("total_time_ms"), "total_time_ms")
    for name, m in (ref_single or {}).items():
        add(f"{name} (single)", m.get("inference_time_per_snapshot_ms"), "ref_single.inference_time_per_snapshot_ms")
    for name, m in (ref_batch or {}).items():
        add(f"{name} (batch)", m.get("inference_time_per_snapshot_ms"), "ref_batch.inference_time_per_snapshot_ms")
    for p, s in (timing_s or {}).items():
        add(f"pipeline (incl. pandas): {p}", 1000.0 * s / n_cases / n_snapshots if n_cases else float("nan"),
            "_timing_s")
    return pd.DataFrame(rows, columns=["method", "ms_per_snapshot", "ms_per_contingency", "source_key"])


def build_contingency_report(assess, no_solution_assess, rows_df, base_nets, runs, n1_nets, ref_single, ref_batch):
    """Dict of DataFrames (all D5 metrics + outage summary + no-solution flags + N-1-vs-base error + cost)."""
    truth_labels = case_snapshot_labels(*assess["truth"]) # extract case-snapshot level truth labels from the assessment
    n_cases = int(truth_labels["case_id"].nunique()) # 
    n_snapshots = len(truth_labels) / n_cases if n_cases else float("nan")   # mean snapshots per solved case
    reg, raw = n1_vs_base_regression(runs, *restrict_nets_to_base_ok(n1_nets, rows_df, base_nets))   # [STSI 151026]:task 093 D-C
    return dict(
        outage_summary=outage_summary(rows_df, truth_labels, base_nets), # only summarizes the outages shares and severity and does not indicate predictor capability
        case_level=case_level_metrics(assess, no_solution_assess),
        element_level=element_level_metrics(assess),
        no_solution_flags=no_solution_flag_table(no_solution_assess),
        severity_ranking=severity_ranking(assess),
        pr_auc=pr_auc_table(assess),
        near_limit_mae=near_limit_mae(assess),
        regression_n1_vs_base=reg,
        cost=cost_table({k: m for (k, s), m in raw.items() if s == "n1"}, ref_single, ref_batch,
                        assess.get("_timing_s", {}), n_cases, n_snapshots),
    )


def _by_predictor(df, columns, values):
    return df.pivot(index="predictor", columns=columns, values=values)


def print_contingency_report(r):
    """HEADLINE first (case-level recall/miss rate + element-level F1, per predictor), then all tables."""
    cl, el = r["case_level"], r["element_level"]
    bar = "=" * 100
    with pd.option_context("display.width", 250, "display.max_columns", 200, "display.max_rows", 2000,
                           "display.float_format", "{:.4f}".format):
        print(bar)
        print("HEADLINE 1 - case-level recall / miss rate (system=ALL); 'any' = solved case-snapshots only")
        a = cl[(cl["system"] == "ALL") & (cl["scope"] == "any")]
        b = cl[(cl["system"] == "ALL") & (cl["scope"] == "any+no_solution")]
        parts = {name: _by_predictor(d, "level", ["recall", "miss_rate"])
                 for name, d in (("any (solved)", a), ("any+no_solution", b)) if len(d)}
        print(pd.concat(parts, axis=1).to_string() if parts else "  (no case-level rows)")
        if len(b):
            print(f"  {NO_SOLUTION_CAVEAT}")
        print("HEADLINE 2 - element-level F1 (system=ALL)")
        e = el[el["system"] == "ALL"]
        print(_by_predictor(e, ["violation_type", "level"], "f1").to_string() if len(e) else "  (no element rows)")
        print(bar)
        titles = dict(outage_summary="OUTAGE SUMMARY (truth side; islanding = bridge outages, "
                                     "no-solution = severe except base_nonconverged snapshots)",   # [STSI 151026]:task 093
                      no_solution_flags="NO-SOLUTION FLAG TABLE")
        order = ["outage_summary", "no_solution_flags", *[k for k in r if k not in ("outage_summary", "no_solution_flags")]]
        for name in order:
            print(f"\n--- {titles.get(name, name.upper())} ---")
            print(r[name].to_string(index=False) if len(r[name]) else "  (empty)")
        print(f"\n{D3_CAVEAT}")


def save_contingency_report(r, save_dir):
    """One CSV per table into a NEW folder; returns the folder used. Never raises on / overwrites an existing path.
    Relative save_dir -> under TRAINING_RESULTS_DIR. If taken: _YYYYMMDD_HHMM, then _YYYYMMDD_HHMMSS, then ..._2, _3, ..."""
    # [STSI 191026]:task 094 - an existing folder lost a 15 h run; never fail at save time, never overwrite
    if not os.path.isabs(save_dir):
        save_dir = os.path.join(TRAINING_RESULTS_DIR, save_dir)
    save_dir = os.path.normpath(save_dir)
    now = datetime.now()
    stamp_s = f"{save_dir}_{now:%Y%m%d_%H%M%S}"
    candidates = [save_dir, f"{save_dir}_{now:%Y%m%d_%H%M}", stamp_s]
    k = 1
    while True:
        if candidates:
            target = candidates.pop(0)
        else:
            k += 1
            target = f"{stamp_s}_{k}"
        try:
            os.makedirs(target)   # exist_ok=False: atomic claim of a fresh folder
            break
        except FileExistsError:
            pass
    for name, df in r.items():
        df.to_csv(os.path.join(target, f"{name}.csv"), index=False)
    note = "" if target == save_dir else f" (requested {save_dir} already existed)"
    print(f"Saved {len(r)} CSVs to {target}{note}")
    return target

In [ ]:
# load the runs list and get the run keys
run_keys=[
    "adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h128_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.1_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    "adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h128_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.1_amboth_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    "adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h64_L3_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.1_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    "adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h128_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.2_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    #"adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h128_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.2_amboth_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    #"adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h64_L3_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.2_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10",

    #"adap-fp0.1_ft0.3_flow-acL0.2_kvl0.2_bs64_lr1.00e-04_h64_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-plain_amedge_delta_vmresidual_phys80_ptdf60-85_acfL45_kvl0",

    #"adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.1-acG0.1_kvl0.7_balDC_bs64_lr1.00e-04_h64_L3_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.2_amedge_delta_vmresidual_phys0_ptdf1_dcfL20-65_dcfG40-75_acfL0_acfG0_kvl0_ramp10",

    #"adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h64_L6_gatv2_manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.2_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_acfL50_acfG80_kvl0_ramp10"
]


In [ ]:
# ==== CONTINGENCY: USER CONFIG (task 095) ====
# [STSI 191026]:task 095 - user run settings (moved verbatim from the driver); the driver reads them via globals().get
path= os.path.join(TRAINING_NETWORKS_DIR,"n1_ieee57_1500_w_Test_57.json")
sweep_paths = [os.path.join(TRAINING_RESULTS_DIR,"best_runs_57.pkl")]   # must be a list; load_runs_by_key iterates it
N1_INDEX_PATH = path      # path to n1_<srcTag>_<desc>.json (from DataGen task 087) 
RUN_KEYS      = run_keys      # D6: REQUIRED list of run keys (no default model)
SWEEP_PATHS   = sweep_paths      # list of sweep/run pickle files that contain RUN_KEYS
REF_SUBSET_N  = 20        # D4(c) timing subset size
RESULTS_SAVE_DIR = "Contingency_Results_57"   # optional NEW folder for CSVs

In [ ]:
# ==== CONTINGENCY: DRIVER (task 090) ====

# [STSI 141026]:end-to-end N-1 evaluation; inert unless run_contingency_eval=True; N1_INDEX_PATH / RUN_KEYS / SWEEP_PATHS set at call time (D6/D7)
# [STSI 191026]:task 095 - values come from the USER CONFIG cell above; None / defaults if it was not run
N1_INDEX_PATH = globals().get("N1_INDEX_PATH", None)        # path to n1_<srcTag>_<desc>.json (from DataGen task 087)
RUN_KEYS      = globals().get("RUN_KEYS", None)             # D6: REQUIRED list of run keys (no default model)
SWEEP_PATHS   = globals().get("SWEEP_PATHS", None)          # list of sweep/run pickle files that contain RUN_KEYS
REF_SUBSET_N  = globals().get("REF_SUBSET_N", 20)           # D4(c) timing subset size
RESULTS_SAVE_DIR = globals().get("RESULTS_SAVE_DIR", None)  # optional folder for CSVs (task 094: never overwrites)
if globals().get("run_contingency_eval", False):   # [STSI 141026]:088 toggles cell is the single switch; local toggle removed, inert if toggles cell not run
    n1_nets, n1_header, n1_rows = load_n1_dataset(N1_INDEX_PATH)
    base_nets = load_source_base_nets(n1_rows)
    runs = load_runs_by_key(RUN_KEYS, SWEEP_PATHS)
    assess = run_violation_assessment(n1_nets, n1_rows, base_nets, runs)
    no_sol = run_no_solution_assessment(n1_rows, base_nets, n1_header, runs)   # task 091, D10b(i)=(c)
    ref_single, ref_batch = run_reference_timing(n1_nets, REF_SUBSET_N) if globals().get("run_ref_timing", False) else ({}, {})
    report = build_contingency_report(assess, no_sol, n1_rows, base_nets, runs, n1_nets, ref_single, ref_batch)
    print_contingency_report(report)
    if RESULTS_SAVE_DIR:
        save_contingency_report(report, RESULTS_SAVE_DIR)